# Trace the Ace — Phase 1 Data Foundation
## Notebook 03 — Data Integrity & Canonical Freeze

**Question:** Can we trust and freeze the reconstructed response/session/turn foundation?

This notebook is the final Phase-1 certification layer. It independently verifies lineage, keys, relationships, sequence integrity, exact duplicates, frozen folds, raw traceability, inference symmetry, and serialization before any canonical artifact is published.

**Hard rule:** candidate data is never promoted to canonical if a blocking integrity check fails. Expected data/audit failures are reported as `passed=False` with an explicit cause; they do not terminate notebook execution.

**Scope boundary:** no retrieval, embeddings, target priors, model features, OOF predictions, calibration, semantic clustering, or mastery interpretation are created here.

# 3.0 — Integrity Bootstrap & Lineage Verification

This section discovers the registered project root, loads the frozen Phase-1 contracts, and verifies that Notebook 1 and Notebook 2 artifacts are still the exact certified inputs.
It does not trust notebook-memory variables from earlier runs.
The inventory manifest binds `responses_base.parquet` to the data contract and source fingerprints.
The parser manifest binds the candidate turn/session Parquet files to their certified hashes and parser gates.
The frozen fold file is verified against both the inventory manifest and the data contract.
Parser-manifest payload integrity is checked when its self-hash is available.
Version checks are enforced when both sides explicitly record the version.
All path resolution is registry-driven; no machine-specific project path is hard-coded.
A failed lineage check disables every later publication gate but execution continues for diagnosis.

**Revision 1.2:** a stale inventory-recorded contract file hash or contract-version advance is non-blocking only when current source fingerprints, response schema, frozen folds, label-blind restrictions, parser normalization policy, and certified parser artifacts independently reconcile. Any authoritative source, schema, policy, fold, or artifact mismatch remains a blocker.


In [1]:
# ============================================================
# 3.0 — INTEGRITY BOOTSTRAP & LINEAGE VERIFICATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os, json, csv, gc, shutil, hashlib, unicodedata, sqlite3, tempfile, platform
import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.compute as pc
    import pyarrow.dataset as ds
    PYARROW_READY = True
    PYARROW_IMPORT_ERROR = ""
except Exception as e:
    pa = pq = pc = ds = None
    PYARROW_READY = False
    PYARROW_IMPORT_ERROR = f"{type(e).__name__}: {e}"

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

NOTEBOOK_VERSION = "1.3"
INTEGRITY_RUN_ID = "INT_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
GLOBAL_ISSUES = []

def check(section, check_name, passed, observed="", expected="", severity="BLOCKER", cause=""):
    passed = bool(passed)
    sev = "" if passed else str(severity).upper()
    status = "PASS" if passed else ("WARN" if sev == "WARNING" else ("INFO" if sev == "INFO" else "FAIL"))
    return {
        "section": section, "check": check_name, "passed": passed,
        "status": status, "severity": sev,
        "observed": str(observed), "expected": str(expected), "cause": str(cause)
    }

def add_issue(section, code, severity, count=1, detail=""):
    GLOBAL_ISSUES.append({
        "section": str(section), "code": str(code), "severity": str(severity),
        "count": int(count), "detail": str(detail)
    })

def get_nested(obj, path, default=None):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def sha256_json_payload(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def json_safe(x):
    if isinstance(x, dict): return {str(k): json_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, set)): return [json_safe(v) for v in x]
    if isinstance(x, Path): return x.as_posix()
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return None if np.isnan(x) else float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, pd.Timestamp): return x.isoformat()
    if isinstance(x, float) and np.isnan(x): return None
    return x

def atomic_json_write(path, payload):
    path = Path(path)
    tmp = path.with_name("." + path.name + ".tmp")
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        if tmp.exists(): tmp.unlink()
        with open(tmp, "w", encoding="utf-8", newline="\n") as f:
            json.dump(json_safe(payload), f, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)
            f.write("\n")
        with open(tmp, "r", encoding="utf-8") as f:
            reread = json.load(f)
        if reread != json_safe(payload):
            return False, "JSON read-back differs from written payload."
        os.replace(tmp, path)
        return True, ""
    except Exception as e:
        try:
            if tmp.exists(): tmp.unlink()
        except Exception:
            pass
        return False, f"{type(e).__name__}: {e}"

def discover_project_root():
    env = os.environ.get("TRACE_THE_ACE_PROJECT_ROOT")
    candidates = []
    if env:
        candidates.append(Path(env).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for root in candidates:
        reg = root / "scratch_mastery_outputs" / "00_project_setup" / "path_registry.json"
        if reg.exists():
            return root, reg
    return None, None

# Safe defaults let later cells diagnose rather than crash.
PROJECT_ROOT = DATA_ROOT = SCRATCH_OUTPUT_ROOT = SETUP_OUTPUT_DIR = None
FROZEN_FOLD_MANIFEST_PATH = TRANSCRIPT_ROOT = None
FOUNDATION_ROOT = INVENTORY_DIR = PARSER_DIR = INTEGRITY_DIR = CANONICAL_DIR = None
DATA_CONTRACT_PATH = INVENTORY_MANIFEST_PATH = RESPONSES_BASE_PATH = None
PARSER_MANIFEST_PATH = TURNS_CANDIDATE_PATH = SESSIONS_CANDIDATE_PATH = None

path_registry = {}
data_contract = {}
inventory_manifest = {}
parser_manifest = {}
parser_payload = {}
responses_base = pd.DataFrame()
sessions_candidate = pd.DataFrame()
frozen_folds = pd.DataFrame()

# Contract lineage can be semantically reconciled when a stale upstream file-hash
# remains after a documented contract amendment, but only if every authoritative
# source binding and frozen artifact still agrees.
CONTRACT_HASH_EXACT = False
CONTRACT_LINEAGE_RECONCILED = False
CONTRACT_LINEAGE_DETAIL = {}

bootstrap_rows = []
root, registry_path = discover_project_root()

bootstrap_rows.append(check("3.0", "PyArrow available", PYARROW_READY, PYARROW_READY, True,
                            cause=PYARROW_IMPORT_ERROR))

if root is None:
    bootstrap_rows.append(check(
        "3.0", "Registered project root discovered", False, "Not found",
        "Project containing scratch_mastery_outputs/00_project_setup/path_registry.json",
        cause="Open/run this notebook from the project tree or set TRACE_THE_ACE_PROJECT_ROOT."
    ))
else:
    try:
        path_registry = load_json(registry_path)
        PROJECT_ROOT = Path(path_registry["project_root"])
        DATA_ROOT = Path(path_registry["data_root"])
        SCRATCH_OUTPUT_ROOT = Path(path_registry["scratch_output_root"])
        SETUP_OUTPUT_DIR = Path(path_registry["setup_output_dir"])
        FROZEN_FOLD_MANIFEST_PATH = Path(path_registry["frozen_fold_manifest_path"])
        TRANSCRIPT_ROOT = Path(path_registry["transcript_source"]) if path_registry.get("transcript_source_type") == "external_path" else None

        FOUNDATION_ROOT = SCRATCH_OUTPUT_ROOT / "01_data_foundation"
        INVENTORY_DIR = FOUNDATION_ROOT / "01_inventory"
        PARSER_DIR = FOUNDATION_ROOT / "02_turn_parser"
        INTEGRITY_DIR = FOUNDATION_ROOT / "03_integrity"
        CANONICAL_DIR = INTEGRITY_DIR / "canonical"

        DATA_CONTRACT_PATH = FOUNDATION_ROOT / "data_contract.json"
        INVENTORY_MANIFEST_PATH = INVENTORY_DIR / "inventory_manifest.json"
        RESPONSES_BASE_PATH = INVENTORY_DIR / "responses_base.parquet"
        PARSER_MANIFEST_PATH = PARSER_DIR / "parser_manifest.json"
        TURNS_CANDIDATE_PATH = PARSER_DIR / "turns_candidate.parquet"
        SESSIONS_CANDIDATE_PATH = PARSER_DIR / "sessions_candidate.parquet"

        bootstrap_rows.append(check("3.0", "Registered project root discovered", PROJECT_ROOT.exists(),
                                    PROJECT_ROOT, "Existing registered project_root"))
        bootstrap_rows.append(check("3.0", "Registry project root matches discovered root",
                                    PROJECT_ROOT.resolve() == root.resolve(), PROJECT_ROOT, root,
                                    cause="Notebook location and path_registry.json resolve to different project roots."))

        required_paths = {
            "data_contract.json": DATA_CONTRACT_PATH,
            "inventory_manifest.json": INVENTORY_MANIFEST_PATH,
            "responses_base.parquet": RESPONSES_BASE_PATH,
            "parser_manifest.json": PARSER_MANIFEST_PATH,
            "turns_candidate.parquet": TURNS_CANDIDATE_PATH,
            "sessions_candidate.parquet": SESSIONS_CANDIDATE_PATH,
            "frozen_fold_manifest.parquet": FROZEN_FOLD_MANIFEST_PATH,
        }
        for name, path in required_paths.items():
            bootstrap_rows.append(check("3.0", f"Required input exists: {name}", path.exists(),
                                        path, "Existing file", cause="Required frozen Phase-1 input is missing."))

        if all(p.exists() for p in required_paths.values()) and PYARROW_READY:
            data_contract = load_json(DATA_CONTRACT_PATH)
            inventory_manifest = load_json(INVENTORY_MANIFEST_PATH)
            parser_manifest = load_json(PARSER_MANIFEST_PATH)
            parser_payload = parser_manifest.get("manifest_payload", parser_manifest)

            responses_base = pq.read_table(RESPONSES_BASE_PATH).to_pandas()
            sessions_candidate = pq.read_table(SESSIONS_CANDIDATE_PATH).to_pandas()
            frozen_folds = pq.read_table(FROZEN_FOLD_MANIFEST_PATH).to_pandas()

            inventory_ready = bool(get_nested(inventory_manifest, ["inventory", "inventory_ready"], False))
            inventory_blockers = int(inventory_manifest.get("blocking_failures", -1))
            bootstrap_rows.append(check("3.0", "Inventory manifest certified", inventory_ready and inventory_blockers == 0,
                                        f"ready={inventory_ready}, blockers={inventory_blockers}", "ready=True, blockers=0"))

            parser_cert = parser_payload.get("certification", {})
            det_cert = parser_cert.get("determinism", {}) if isinstance(parser_cert, dict) else {}
            parser_ready = all([
                parser_cert.get("integrated_parser_ready") is True,
                parser_cert.get("production_parse_ready") is True,
                parser_cert.get("corpus_parser_audit_ready") is True,
                parser_cert.get("determinism_reconstruction_ready") is True,
                parser_cert.get("corpus_required_failures") in (0, None),
                det_cert.get("failed_sessions") in (0, None),
                det_cert.get("failed_checks") in (0, None),
            ])
            bootstrap_rows.append(check("3.0", "Parser manifest upstream certifications clean", parser_ready,
                                        parser_cert, "All four parser certifications True"))

            candidate_boundary = (
                get_nested(parser_payload, ["identity", "artifact_status"]) == "CANDIDATE"
                and get_nested(parser_payload, ["identity", "canonical"]) is False
            )
            bootstrap_rows.append(check("3.0", "Parser artifacts explicitly non-canonical", candidate_boundary,
                                        f"status={get_nested(parser_payload,['identity','artifact_status'])}, "
                                        f"canonical={get_nested(parser_payload,['identity','canonical'])}",
                                        "CANDIDATE / False"))

            # Manifest self-integrity.
            stored_payload_hash = parser_manifest.get("manifest_payload_sha256")
            if stored_payload_hash and "manifest_payload" in parser_manifest:
                actual_payload_hash = sha256_json_payload(parser_manifest["manifest_payload"])
                bootstrap_rows.append(check("3.0", "Parser manifest payload SHA256 verifies",
                                            actual_payload_hash == stored_payload_hash,
                                            actual_payload_hash, stored_payload_hash,
                                            cause="parser_manifest.json content no longer matches its payload hash."))
            else:
                bootstrap_rows.append(check("3.0", "Parser manifest payload SHA256 available",
                                            True, "Not recorded", "Optional for legacy manifest",
                                            severity="WARNING"))
                add_issue("3.0", "PARSER_MANIFEST_SELF_HASH_NOT_RECORDED", "WARNING", 1,
                          "Legacy parser manifest has no payload self-hash.")

            # Inventory-bound artifacts. responses_base is immutable and must match exactly.
            # data_contract.json is handled more carefully: an old recorded file hash may become
            # stale after a documented 1.0→1.1 contract amendment. We never silently accept it.
            # Instead, a hash mismatch is downgraded to WARNING only when the current contract
            # independently reconciles with every authoritative inventory/source binding.
            expected_contract_hash = get_nested(inventory_manifest, ["artifacts", "data_contract_sha256"])
            expected_response_hash = get_nested(inventory_manifest, ["artifacts", "responses_base_sha256"])
            current_contract_hash = sha256_file(DATA_CONTRACT_PATH)
            current_response_hash = sha256_file(RESPONSES_BASE_PATH)

            response_hash_ok = bool(expected_response_hash) and current_response_hash == expected_response_hash
            bootstrap_rows.append(check("3.0", "responses_base hash matches inventory freeze",
                                        response_hash_ok, current_response_hash, expected_response_hash,
                                        cause="responses_base.parquet changed after Notebook 1 freeze."))

            inv_pop = inventory_manifest.get("population", {}) if isinstance(inventory_manifest, dict) else {}
            contract_source = get_nested(data_contract, ["source_binding"], {}) or {}
            inv_source = get_nested(inventory_manifest, ["source_fingerprints"], {}) or {}
            contract_fold = get_nested(data_contract, ["fold_contract"], {}) or {}
            inv_folds = get_nested(inventory_manifest, ["folds"], {}) or {}
            contract_resp_schema = get_nested(data_contract, ["response_schema"], {}) or {}
            inv_contract_version = str(get_nested(inventory_manifest, ["inventory", "contract_version"], ""))
            current_contract_version = str(get_nested(data_contract, ["contract", "version"], ""))

            source_pairs = {
                "train_features_sha256": (
                    contract_source.get("train_features_sha256"), inv_source.get("train_features_sha256")
                ),
                "train_labels_sha256": (
                    contract_source.get("train_labels_sha256"), inv_source.get("train_labels_sha256")
                ),
                "transcript_directory_sha256": (
                    contract_source.get("transcript_directory_sha256"), inv_source.get("transcript_directory_sha256")
                ),
                "frozen_fold_manifest_sha256": (
                    contract_source.get("frozen_fold_manifest_sha256"),
                    inv_source.get("frozen_fold_manifest_sha256")
                ),
            }
            source_pair_checks = {
                k: (bool(a) and bool(b) and str(a) == str(b)) for k, (a, b) in source_pairs.items()
            }

            population_checks = {
                "responses": int(contract_source.get("expected_response_count", -1)) == int(inv_pop.get("responses", -2)),
                "sessions": int(contract_source.get("expected_session_count", -1)) == int(inv_pop.get("sessions", -2)),
                "transcript_files": int(contract_source.get("expected_transcript_file_count", -1))
                    == int(get_nested(inventory_manifest, ["transcripts", "files"], -2)),
            }

            fold_contract_ok = (
                str(contract_fold.get("manifest_sha256", ""))
                == str(inv_folds.get("manifest_sha256", ""))
                == str(inv_source.get("frozen_fold_manifest_sha256", ""))
                and bool(contract_fold.get("session_grouped", False))
                and get_nested(data_contract, ["fold_contract", "cross_fold_session_allowed"], None) is False
            )

            response_schema_ok = (
                set(contract_resp_schema.get("required_fields", [])) <= set(responses_base.columns)
                and contract_resp_schema.get("response_key", "response_id") in responses_base.columns
                and contract_resp_schema.get("session_key", "session_id") in responses_base.columns
                and contract_resp_schema.get("target_field", "target") in responses_base.columns
                and contract_resp_schema.get("fold_field", "fold") in responses_base.columns
            )

            label_blind_ok = get_nested(data_contract, ["parser_restrictions", "label_blind"], False) is True

            # Contract version may legitimately advance after Notebook 1 (for example the
            # parser-safe NFC amendment). Version-string drift alone is not source drift.
            # We certify the current contract from invariant source/schema/fold/policy bindings.
            contract_unicode_now = str(get_nested(data_contract, ["text_normalization", "unicode_form"], "")).upper()
            parser_unicode_now = str(get_nested(parser_payload, ["parser_policy", "text_normalization", "unicode_form"], "")).upper()
            normalization_policy_ok = bool(contract_unicode_now) and bool(parser_unicode_now) and contract_unicode_now == parser_unicode_now

            version_metadata_present = bool(current_contract_version)
            version_changed_after_inventory = bool(
                inv_contract_version and current_contract_version
                and inv_contract_version != current_contract_version
            )

            CONTRACT_HASH_EXACT = bool(expected_contract_hash) and current_contract_hash == expected_contract_hash
            CONTRACT_LINEAGE_RECONCILED = bool(
                response_hash_ok
                and all(source_pair_checks.values())
                and all(population_checks.values())
                and fold_contract_ok
                and response_schema_ok
                and label_blind_ok
                and normalization_policy_ok
                and version_metadata_present
            )
            CONTRACT_LINEAGE_DETAIL = {
                "exact_file_hash_match": CONTRACT_HASH_EXACT,
                "inventory_recorded_sha256": expected_contract_hash,
                "current_sha256": current_contract_hash,
                "inventory_contract_version": inv_contract_version,
                "current_contract_version": current_contract_version,
                "version_changed_after_inventory": version_changed_after_inventory,
                "source_bindings": source_pair_checks,
                "population_bindings": population_checks,
                "fold_contract": fold_contract_ok,
                "response_schema": response_schema_ok,
                "label_blind": label_blind_ok,
                "normalization_policy_matches_parser": normalization_policy_ok,
            }

            if version_changed_after_inventory:
                add_issue(
                    "3.0", "CONTRACT_VERSION_ADVANCED_AFTER_INVENTORY", "WARNING", 1,
                    f"inventory_contract_version={inv_contract_version}; current_contract_version={current_contract_version}. "
                    "Version drift is accepted only because authoritative source/schema/fold/parser-policy bindings are independently re-certified."
                )

            if CONTRACT_HASH_EXACT:
                bootstrap_rows.append(check("3.0", "Data contract file SHA256 matches inventory record",
                                            True, current_contract_hash, expected_contract_hash))
            else:
                mismatch_severity = "WARNING" if CONTRACT_LINEAGE_RECONCILED else "BLOCKER"
                mismatch_cause = (
                    "Inventory-recorded data_contract file SHA256 is stale, but the current contract "
                    "reconciles exactly with certified response, source, fold, schema, and label-blind bindings."
                    if CONTRACT_LINEAGE_RECONCILED else
                    "data_contract file SHA256 differs and semantic lineage could not be fully reconciled."
                )
                bootstrap_rows.append(check("3.0", "Data contract file SHA256 matches inventory record",
                                            False, current_contract_hash, expected_contract_hash,
                                            severity=mismatch_severity, cause=mismatch_cause))
                if CONTRACT_LINEAGE_RECONCILED:
                    add_issue("3.0", "STALE_INVENTORY_DATA_CONTRACT_HASH", "WARNING", 1,
                              f"inventory={expected_contract_hash}; current={current_contract_hash}; "
                              "semantic/source lineage independently reconciled.")

            bootstrap_rows.append(check("3.0", "Current data contract semantic/source lineage reconciles",
                                        CONTRACT_LINEAGE_RECONCILED,
                                        json.dumps(CONTRACT_LINEAGE_DETAIL, sort_keys=True),
                                        "All authoritative bindings agree",
                                        cause="One or more current contract bindings disagree with frozen inventory/source truth."))

            # Candidate hashes from current parser manifest; support both new and legacy shapes.
            expected_turn_hash = (
                get_nested(parser_payload, ["artifacts", "turns_candidate", "sha256"])
                or parser_manifest.get("turns_candidate_sha256")
            )
            expected_session_hash = (
                get_nested(parser_payload, ["artifacts", "sessions_candidate", "sha256"])
                or parser_manifest.get("sessions_candidate_sha256")
            )
            current_turn_hash = sha256_file(TURNS_CANDIDATE_PATH)
            current_session_hash = sha256_file(SESSIONS_CANDIDATE_PATH)
            bootstrap_rows.append(check("3.0", "Turn candidate hash matches parser freeze",
                                        bool(expected_turn_hash) and current_turn_hash == expected_turn_hash,
                                        current_turn_hash, expected_turn_hash,
                                        cause="turns_candidate.parquet differs from certified parser output."))
            bootstrap_rows.append(check("3.0", "Session candidate hash matches parser freeze",
                                        bool(expected_session_hash) and current_session_hash == expected_session_hash,
                                        current_session_hash, expected_session_hash,
                                        cause="sessions_candidate.parquet differs from certified parser output."))

            # Fold-file binding.
            expected_fold_hashes = {
                str(get_nested(data_contract, ["fold_contract", "manifest_sha256"], "")),
                str(get_nested(data_contract, ["source_binding", "frozen_fold_manifest_sha256"], "")),
                str(get_nested(inventory_manifest, ["folds", "manifest_sha256"], "")),
                str(get_nested(inventory_manifest, ["source_fingerprints", "frozen_fold_manifest_sha256"], "")),
            }
            expected_fold_hashes.discard("")
            current_fold_hash = sha256_file(FROZEN_FOLD_MANIFEST_PATH)
            fold_hash_ok = bool(expected_fold_hashes) and expected_fold_hashes == {current_fold_hash}
            bootstrap_rows.append(check("3.0", "Frozen fold fingerprint matches all recorded contracts",
                                        fold_hash_ok, current_fold_hash, sorted(expected_fold_hashes),
                                        cause="Frozen fold artifact or its recorded binding changed."))

            # Parser/version compatibility. Some early parser manifests omitted the explicit
            # contract-version field; that omission is informational when policies/source bindings agree.
            contract_version = str(get_nested(data_contract, ["contract", "version"], ""))
            parser_contract_version = get_nested(parser_payload, ["identity", "data_contract_version"])
            version_ok = parser_contract_version in (None, "", contract_version)
            bootstrap_rows.append(check("3.0", "Parser/data-contract versions compatible", version_ok,
                                        parser_contract_version, contract_version,
                                        cause="Parser manifest explicitly records a different data-contract version."))
            if parser_contract_version in (None, ""):
                add_issue("3.0", "PARSER_CONTRACT_VERSION_NOT_RECORDED", "INFO", 1,
                          "Compatibility is established from certified parser policy/source bindings instead.")

            # Implemented text-normalization policy must agree when both contracts record it.
            contract_unicode = get_nested(data_contract, ["text_normalization", "unicode_form"])
            parser_unicode = get_nested(parser_payload, ["parser_policy", "text_normalization", "unicode_form"])
            unicode_ok = (not contract_unicode or not parser_unicode or str(contract_unicode).upper() == str(parser_unicode).upper())
            bootstrap_rows.append(check("3.0", "Recorded Unicode normalization policies agree", unicode_ok,
                                        parser_unicode, contract_unicode,
                                        cause="Parser implementation policy and current data contract disagree."))

    except Exception as e:
        bootstrap_rows.append(check("3.0", "Bootstrap loading completed", False, type(e).__name__,
                                    "Successful load", cause=f"{type(e).__name__}: {e}"))

# Keep later diagnostic/report cells executable even if project discovery failed.
if INTEGRITY_DIR is None:
    INTEGRITY_DIR = Path.cwd() / "_trace_the_ace_integrity_diagnostics"
if CANONICAL_DIR is None:
    CANONICAL_DIR = INTEGRITY_DIR / "canonical"
if FOUNDATION_ROOT is None:
    FOUNDATION_ROOT = INTEGRITY_DIR.parent

bootstrap_audit = pd.DataFrame(bootstrap_rows)
INTEGRITY_BOOTSTRAP_READY = bool(len(bootstrap_audit)) and bootstrap_audit.loc[
    bootstrap_audit["severity"].eq("BLOCKER") | bootstrap_audit["severity"].eq(""), "passed"
].all()

display(bootstrap_audit)
print("\n" + "=" * 76)
print("TRACE THE ACE — INTEGRITY BOOTSTRAP")
print("=" * 76)
print(f"Run ID                    : {INTEGRITY_RUN_ID}")
print(f"Project root              : {PROJECT_ROOT}")
print(f"Bootstrap blocking fails  : {int((~bootstrap_audit['passed'] & bootstrap_audit['severity'].eq('BLOCKER')).sum())}")
print(f"INTEGRITY BOOTSTRAP READY : {INTEGRITY_BOOTSTRAP_READY}")
print("=" * 76)

,section,check,passed,status,severity,observed,expected,cause
0,3.0,PyArrow available,True,PASS,,True,True,
1,3.0,Registered project root discovered,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing registered project_root,
2,3.0,Registry project root matches discovered root,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Notebook location and path_registry.json resol...
3,3.0,Required input exists: data_contract.json,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
4,3.0,Required input exists: inventory_manifest.json,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
5,3.0,Required input exists: responses_base.parquet,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
6,3.0,Required input exists: parser_manifest.json,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
7,3.0,Required input exists: turns_candidate.parquet,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
8,3.0,Required input exists: sessions_candidate.parquet,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.
9,3.0,Required input exists: frozen_fold_manifest.pa...,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Existing file,Required frozen Phase-1 input is missing.



TRACE THE ACE — INTEGRITY BOOTSTRAP
Run ID                    : INT_20260811T191628Z
Project root              : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
Bootstrap blocking fails  : 0
INTEGRITY BOOTSTRAP READY : True


# 3.1 — Entity, Schema & Key Integrity

This section certifies the four Phase-1 entities independently: responses, sessions, turns, and frozen fold assignments.
Response validation follows the field names frozen in `data_contract.json`.
Session keys and candidate schemas are checked against the parser manifest rather than a manually remembered schema.
Turn identity uniqueness is measured exactly one identity column at a time to keep memory bounded; a disk-backed SQLite fallback is available if Arrow distinct counting cannot fit.
The natural turn key `(session_id, turn_index)` is certified again in Section 3.2 during the exact session-stream scan.
The frozen fold manifest is never regenerated.
Any invalid response key, target, fold, candidate schema, session key, or turn identity is a blocker.
No target-dependent statistic is created in this section.

In [2]:
# ============================================================
# 3.1 — ENTITY, SCHEMA & KEY INTEGRITY
# ============================================================

def parquet_distinct_count_exact(path, column, tmp_dir):
    """Exact distinct count. Arrow first; disk-backed SQLite fallback."""
    try:
        arr = pq.read_table(path, columns=[column]).column(0)
        value = int(pc.count_distinct(arr).as_py())
        del arr
        gc.collect()
        return value, "PYARROW", ""
    except Exception as first_error:
        db_path = Path(tmp_dir) / f".distinct_{column}_{INTEGRITY_RUN_ID}.sqlite"
        conn = None
        try:
            if db_path.exists(): db_path.unlink()
            conn = sqlite3.connect(db_path)
            conn.execute("PRAGMA journal_mode=OFF")
            conn.execute("PRAGMA synchronous=OFF")
            conn.execute("PRAGMA temp_store=MEMORY")
            conn.execute("CREATE TABLE u (v TEXT PRIMARY KEY)")
            pf = pq.ParquetFile(path)
            for batch in pf.iter_batches(columns=[column], batch_size=200_000):
                vals = batch.column(0).to_pylist()
                conn.executemany("INSERT OR IGNORE INTO u(v) VALUES (?)",
                                 [(None if v is None else str(v),) for v in vals])
                conn.commit()
            value = int(conn.execute("SELECT COUNT(*) FROM u").fetchone()[0])
            conn.close(); conn = None
            db_path.unlink(missing_ok=True)
            return value, "SQLITE_FALLBACK", f"Arrow path failed first: {type(first_error).__name__}"
        except Exception as second_error:
            try:
                if conn is not None: conn.close()
                db_path.unlink(missing_ok=True)
            except Exception:
                pass
            return None, "FAILED", (
                f"Arrow={type(first_error).__name__}: {first_error}; "
                f"SQLite={type(second_error).__name__}: {second_error}"
            )

entity_rows = []
entity_summary = {}

if not INTEGRITY_BOOTSTRAP_READY:
    entity_rows.append(check("3.1", "Section entry gate", False, False, True,
                             cause="Section 3.0 is not ready."))
else:
    try:
        response_required = list(get_nested(data_contract, ["response_schema", "required_fields"], []))
        response_key = get_nested(data_contract, ["response_schema", "response_key"], "response_id")
        response_session = get_nested(data_contract, ["response_schema", "session_key"], "session_id")
        objective_field = get_nested(data_contract, ["response_schema", "objective_field"], "objective_raw")
        objective_id_field = get_nested(data_contract, ["response_schema", "objective_id_field"], "objective_id_raw")
        target_field = get_nested(data_contract, ["response_schema", "target_field"], "target")
        fold_field = get_nested(data_contract, ["response_schema", "fold_field"], "fold")
        valid_folds = set(get_nested(data_contract, ["fold_contract", "fold_ids"], [0,1,2,3,4]))

        expected_responses = int(get_nested(data_contract, ["source_binding", "expected_response_count"],
                                           get_nested(inventory_manifest, ["population", "responses"], -1)))
        expected_sessions = int(get_nested(data_contract, ["source_binding", "expected_session_count"],
                                          get_nested(inventory_manifest, ["population", "sessions"], -1)))
        expected_turns = int(get_nested(parser_payload, ["source_binding", "expected_logical_rows"],
                                       get_nested(parser_payload, ["artifacts", "turns_candidate", "rows"], -1)))

        response_cols = set(responses_base.columns)
        missing_response = sorted(set(response_required) - response_cols)
        entity_rows.append(check("3.1", "Response required schema present", not missing_response,
                                 sorted(response_cols), response_required,
                                 cause=f"Missing fields: {missing_response}" if missing_response else ""))

        if not missing_response:
            response_nulls = responses_base[response_required].isna().sum().to_dict()
            response_blank_session = int(responses_base[response_session].astype(str).str.strip().eq("").sum())
            response_blank_objective = int(responses_base[objective_field].astype(str).str.strip().eq("").sum())
            response_id_unique = int(responses_base[response_key].nunique(dropna=False))
            target_values = set(pd.to_numeric(responses_base[target_field], errors="coerce").dropna().astype(int).unique())
            fold_values = set(pd.to_numeric(responses_base[fold_field], errors="coerce").dropna().astype(int).unique())

            entity_rows += [
                check("3.1", "Response row census", len(responses_base) == expected_responses,
                      len(responses_base), expected_responses, cause="Response population changed."),
                check("3.1", "response_id non-null", int(responses_base[response_key].isna().sum()) == 0,
                      int(responses_base[response_key].isna().sum()), 0),
                check("3.1", "response_id globally unique", response_id_unique == len(responses_base),
                      response_id_unique, len(responses_base), cause="Duplicate response_id detected."),
                check("3.1", "Response session_id non-blank",
                      int(responses_base[response_session].isna().sum()) == 0 and response_blank_session == 0,
                      f"null={int(responses_base[response_session].isna().sum())}, blank={response_blank_session}", 0),
                check("3.1", "Objective text non-blank",
                      int(responses_base[objective_field].isna().sum()) == 0 and response_blank_objective == 0,
                      f"null={int(responses_base[objective_field].isna().sum())}, blank={response_blank_objective}", 0),
                check("3.1", "Target is binary", target_values.issubset({0,1})
                      and int(pd.to_numeric(responses_base[target_field], errors="coerce").isna().sum()) == 0,
                      sorted(target_values), [0,1], cause="Target contains missing or non-binary values."),
                check("3.1", "Fold values match frozen contract", fold_values == valid_folds
                      and int(pd.to_numeric(responses_base[fold_field], errors="coerce").isna().sum()) == 0,
                      sorted(fold_values), sorted(valid_folds), cause="Response fold values differ from frozen contract."),
            ]

        # Session keys and parser-certified schema.
        session_expected_fields = get_nested(parser_payload, ["schemas", "session_candidate", "field_count"])
        session_schema_fields = len(pq.read_schema(SESSIONS_CANDIDATE_PATH))
        entity_rows += [
            check("3.1", "Session row census", len(sessions_candidate) == expected_sessions,
                  len(sessions_candidate), expected_sessions),
            check("3.1", "Session ID non-null", "session_id" in sessions_candidate
                  and int(sessions_candidate["session_id"].isna().sum()) == 0,
                  int(sessions_candidate["session_id"].isna().sum()) if "session_id" in sessions_candidate else "missing", 0),
            check("3.1", "Session ID unique", "session_id" in sessions_candidate
                  and int(sessions_candidate["session_id"].nunique(dropna=False)) == len(sessions_candidate),
                  int(sessions_candidate["session_id"].nunique(dropna=False)) if "session_id" in sessions_candidate else "missing",
                  len(sessions_candidate)),
            check("3.1", "Session schema matches parser manifest",
                  session_expected_fields is None or int(session_expected_fields) == session_schema_fields,
                  session_schema_fields, session_expected_fields if session_expected_fields is not None else "not recorded"),
        ]

        # Turn schema and identity.
        turn_schema = pq.read_schema(TURNS_CANDIDATE_PATH)
        turn_cols = set(turn_schema.names)
        required_turn_keys = {"session_id", "turn_uid", "source_row_uid", "turn_index",
                              "source_file_relative", "source_row_index", "content_raw"}
        missing_turn_keys = sorted(required_turn_keys - turn_cols)
        turn_expected_fields = get_nested(parser_payload, ["schemas", "turn_candidate", "field_count"])
        turn_rows = int(pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows)
        entity_rows += [
            check("3.1", "Turn required key/provenance fields present", not missing_turn_keys,
                  sorted(turn_cols), sorted(required_turn_keys),
                  cause=f"Missing fields: {missing_turn_keys}" if missing_turn_keys else ""),
            check("3.1", "Turn row census", turn_rows == expected_turns, turn_rows, expected_turns),
            check("3.1", "Turn schema matches parser manifest",
                  turn_expected_fields is None or int(turn_expected_fields) == len(turn_schema),
                  len(turn_schema), turn_expected_fields if turn_expected_fields is not None else "not recorded"),
        ]

        uid_counts = {}
        if not missing_turn_keys:
            for col in ["turn_uid", "source_row_uid"]:
                distinct, method, err = parquet_distinct_count_exact(TURNS_CANDIDATE_PATH, col, INTEGRITY_DIR)
                uid_counts[col] = {"distinct": distinct, "method": method, "error": err}
                entity_rows.append(check("3.1", f"{col} globally unique",
                                         distinct == turn_rows, distinct, turn_rows,
                                         cause=err or f"Duplicate {col} detected."))

        # Frozen fold entity.
        fold_required = {"response_id", "session_id", "fold"}
        missing_fold = sorted(fold_required - set(frozen_folds.columns))
        entity_rows += [
            check("3.1", "Frozen fold schema present", not missing_fold,
                  sorted(frozen_folds.columns), sorted(fold_required),
                  cause=f"Missing fields: {missing_fold}" if missing_fold else ""),
            check("3.1", "Frozen fold row census", len(frozen_folds) == expected_responses,
                  len(frozen_folds), expected_responses),
        ]
        if not missing_fold:
            entity_rows += [
                check("3.1", "Frozen response assignment unique",
                      int(frozen_folds["response_id"].nunique(dropna=False)) == len(frozen_folds),
                      int(frozen_folds["response_id"].nunique(dropna=False)), len(frozen_folds)),
                check("3.1", "Frozen fold required fields non-null",
                      int(frozen_folds[list(fold_required)].isna().sum().sum()) == 0,
                      int(frozen_folds[list(fold_required)].isna().sum().sum()), 0),
            ]

        entity_summary = {
            "responses": len(responses_base), "sessions": len(sessions_candidate),
            "turns": turn_rows, "turn_uid_check": uid_counts
        }

    except Exception as e:
        entity_rows.append(check("3.1", "Entity/key audit completed", False, type(e).__name__,
                                 "Successful audit", cause=f"{type(e).__name__}: {e}"))

entity_key_audit = pd.DataFrame(entity_rows)
ENTITY_KEY_INTEGRITY_READY = bool(len(entity_key_audit)) and not (
    (~entity_key_audit["passed"]) & entity_key_audit["severity"].eq("BLOCKER")
).any()

display(entity_key_audit)
print(f"\nENTITY KEY INTEGRITY READY : {ENTITY_KEY_INTEGRITY_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.1,Response required schema present,True,PASS,,"['fold', 'objective_id_raw', 'objective_raw', ...","['response_id', 'session_id', 'objective_id_ra...",
1,3.1,Response row census,True,PASS,,35072,35072,Response population changed.
2,3.1,response_id non-null,True,PASS,,0,0,
3,3.1,response_id globally unique,True,PASS,,35072,35072,Duplicate response_id detected.
4,3.1,Response session_id non-blank,True,PASS,,"null=0, blank=0",0,
5,3.1,Objective text non-blank,True,PASS,,"null=0, blank=0",0,
6,3.1,Target is binary,True,PASS,,"[np.int64(0), np.int64(1)]","[0, 1]",Target contains missing or non-binary values.
7,3.1,Fold values match frozen contract,True,PASS,,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[0, 1, 2, 3, 4]",Response fold values differ from frozen contract.
8,3.1,Session row census,True,PASS,,22821,22821,
9,3.1,Session ID non-null,True,PASS,,0,0,



ENTITY KEY INTEGRITY READY : True


# 3.2 — Relational Cardinality & Coverage Integrity

This section proves the Phase-1 relational graph: many response rows map to exactly one session, and every session maps to one contiguous block of one or more turns.
No giant response-by-turn denormalized table is created.
The turn file is streamed in batches and processed as contiguous session runs, so only one session-sized block is held for exact sequence checks.
For every session, the scan records actual turn count, exact turn-index uniqueness/contiguity, source-row contiguity, role census, source-file consistency, first/last flags, relative-position validity, and selected quality counters.
A session that reappears after its block has already closed is treated as a blocker.
The streamed session summaries are reconciled against `sessions_candidate.parquet`.
Response coverage, orphan sessions, orphan turns, and silent join expansion are all reported explicitly.
The required labelled response → session → turns chain must have 100% coverage.

In [3]:
# ============================================================
# 3.2 — RELATIONAL CARDINALITY & COVERAGE INTEGRITY
# ============================================================

def summarize_turn_run(g):
    sid = g["session_id"].iloc[0]
    order = np.argsort(pd.to_numeric(g["turn_index"], errors="coerce").to_numpy())
    gs = g.iloc[order]
    ti = pd.to_numeric(gs["turn_index"], errors="coerce").to_numpy()
    si = pd.to_numeric(gs["source_row_index"], errors="coerce").to_numpy()
    n = len(gs)

    turn_index_valid = np.isfinite(ti).all() and np.array_equal(ti.astype(np.int64), np.arange(n, dtype=np.int64))
    source_index_valid = np.isfinite(si).all() and np.array_equal(
        np.sort(si.astype(np.int64)), np.arange(n, dtype=np.int64)
    )

    roles = gs["role"].astype(str).value_counts().to_dict() if "role" in gs else {}
    first_ok = True
    last_ok = True
    if "is_first_turn" in gs:
        vals = gs["is_first_turn"].fillna(False).astype(bool).to_numpy()
        first_ok = int(vals.sum()) == 1 and bool(vals[0])
    if "is_last_turn" in gs:
        vals = gs["is_last_turn"].fillna(False).astype(bool).to_numpy()
        last_ok = int(vals.sum()) == 1 and bool(vals[-1])

    rel_ok = True
    if "relative_turn_position" in gs:
        rel = pd.to_numeric(gs["relative_turn_position"], errors="coerce")
        rel_ok = bool(rel.notna().all() and rel.between(0, 1, inclusive="both").all())

    consecutive_dup = 0
    if "content_hash" in gs and n > 1:
        ch = gs["content_hash"].astype("string").to_numpy()
        consecutive_dup = int(np.sum(ch[1:] == ch[:-1]))

    max_same_role_run = 0
    if "role" in gs and n:
        rr = gs["role"].astype(str).to_numpy()
        switches = np.r_[True, rr[1:] != rr[:-1]]
        run_ids = np.cumsum(switches)
        _, counts = np.unique(run_ids, return_counts=True)
        max_same_role_run = int(counts.max()) if len(counts) else 0

    out = {
        "session_id": sid, "n_turns_actual": n,
        "turn_index_exact": bool(turn_index_valid),
        "source_row_index_exact": bool(source_index_valid),
        "n_student_actual": int(roles.get("student", 0)),
        "n_tutor_actual": int(roles.get("tutor", 0)),
        "n_background_actual": int(roles.get("background", 0)),
        "n_other_actual": int(sum(v for k, v in roles.items() if k not in {"student","tutor","background"})),
        "source_file_nunique": int(gs["source_file_relative"].nunique(dropna=False)) if "source_file_relative" in gs else -1,
        "source_file_relative_actual": str(gs["source_file_relative"].iloc[0]) if "source_file_relative" in gs and n else "",
        "first_flag_exact": bool(first_ok), "last_flag_exact": bool(last_ok),
        "relative_position_valid": bool(rel_ok),
        "consecutive_content_hash_duplicates": int(consecutive_dup),
        "max_same_role_run": int(max_same_role_run),
    }
    return out

def stream_session_runs(path, wanted_columns, batch_size=131_072):
    schema_names = set(pq.read_schema(path).names)
    cols = [c for c in wanted_columns if c in schema_names]
    summaries, reappearances = [], []
    completed = set()
    carry = None
    pf = pq.ParquetFile(path)

    def process_complete(df):
        if df is None or df.empty:
            return
        sid_series = df["session_id"].astype("string")
        boundary = sid_series.ne(sid_series.shift()).fillna(True)
        if len(boundary):
            boundary.iloc[0] = True
        run_id = boundary.cumsum()
        for _, g in df.groupby(run_id, sort=False, dropna=False):
            sid = g["session_id"].iloc[0]
            sid_key = "__NULL__" if pd.isna(sid) else str(sid)
            if sid_key in completed:
                reappearances.append(sid_key)
            else:
                summaries.append(summarize_turn_run(g))
                completed.add(sid_key)

    for batch in pf.iter_batches(columns=cols, batch_size=batch_size):
        df = batch.to_pandas()
        if carry is not None and not carry.empty:
            df = pd.concat([carry, df], ignore_index=True)
        if df.empty:
            continue

        sid_series = df["session_id"].astype("string")
        boundary = sid_series.ne(sid_series.shift()).fillna(True)
        if len(boundary):
            boundary.iloc[0] = True
        run_id = boundary.cumsum()
        last_run = run_id.iloc[-1]
        complete = df.loc[run_id.ne(last_run).fillna(False)]
        carry = df.loc[run_id.eq(last_run).fillna(False)].copy()
        process_complete(complete)

    process_complete(carry)
    return pd.DataFrame(summaries), sorted(set(reappearances))

# Synthetic guard: the first row of every contiguous run must receive a real run ID.
_test_sid = pd.Series(["A", "A", "B", "B"], dtype="string")
_test_boundary = _test_sid.ne(_test_sid.shift()).fillna(True)
_test_boundary.iloc[0] = True
_test_run_id = _test_boundary.cumsum()
STREAM_RUN_ID_SELF_TEST = bool(
    _test_run_id.notna().all() and _test_run_id.tolist() == [1, 1, 2, 2]
)

relation_rows = []
relation_join_log = pd.DataFrame()
turn_session_scan = pd.DataFrame()
session_reappearances = []

if not (INTEGRITY_BOOTSTRAP_READY and ENTITY_KEY_INTEGRITY_READY):
    relation_rows.append(check("3.2", "Section entry gate", False, False, True,
                               cause="Sections 3.0–3.1 are not ready."))
else:
    try:
        relation_rows.append(check(
            "3.2", "Streaming run-boundary self-test",
            STREAM_RUN_ID_SELF_TEST, STREAM_RUN_ID_SELF_TEST, True,
            cause="Batch grouping must never drop the first row of a contiguous session run."
        ))
        scan_columns = [
            "session_id", "turn_index", "source_row_index", "role", "source_file_relative",
            "is_first_turn", "is_last_turn", "relative_turn_position", "content_hash"
        ]
        turn_session_scan, session_reappearances = stream_session_runs(TURNS_CANDIDATE_PATH, scan_columns)

        response_sessions = set(responses_base["session_id"].astype(str))
        session_ids = set(sessions_candidate["session_id"].astype(str))
        turn_sessions = set(turn_session_scan["session_id"].astype(str))

        unmatched_response_sessions = sorted(response_sessions - session_ids)
        sessions_without_turns = sorted(session_ids - turn_sessions)
        orphan_turn_sessions = sorted(turn_sessions - session_ids)
        orphan_unlabelled_sessions = sorted(session_ids - response_sessions)

        # M:1 response -> session is guaranteed only when session key is unique and every response session exists.
        response_join_result_rows = len(responses_base) if not unmatched_response_sessions else (
            len(responses_base) - int(responses_base["session_id"].astype(str).isin(unmatched_response_sessions).sum())
        )

        relation_rows += [
            check("3.2", "Turn sessions are contiguous blocks", len(session_reappearances) == 0,
                  len(session_reappearances), 0,
                  cause=("Sessions reappeared after a completed block: " + ", ".join(session_reappearances[:10]))
                  if session_reappearances else ""),
            check("3.2", "Every response session exists exactly once in session table",
                  len(unmatched_response_sessions) == 0 and sessions_candidate["session_id"].is_unique,
                  len(unmatched_response_sessions), 0),
            check("3.2", "Response→Session join preserves response population",
                  response_join_result_rows == len(responses_base),
                  response_join_result_rows, len(responses_base),
                  cause="Missing or non-unique session key would change response population."),
            check("3.2", "Every candidate session has at least one turn",
                  len(sessions_without_turns) == 0, len(sessions_without_turns), 0),
            check("3.2", "No turn belongs to an unknown candidate session",
                  len(orphan_turn_sessions) == 0, len(orphan_turn_sessions), 0),
            check("3.2", "No unused transcript session relative to labelled response population",
                  len(orphan_unlabelled_sessions) == 0, len(orphan_unlabelled_sessions), 0,
                  severity="BLOCKER",
                  cause="Canonical foundation is expected to match the labelled session population exactly."),
        ]

        # Reconcile streamed counts and role census with sessions_candidate.
        compare_cols = [
            "session_id", "n_turns", "n_student_turns", "n_tutor_turns",
            "n_background_turns", "n_unknown_roles", "source_file_relative"
        ]
        available_compare = [c for c in compare_cols if c in sessions_candidate.columns]
        rec = sessions_candidate[available_compare].merge(turn_session_scan, on="session_id", how="outer",
                                                          validate="one_to_one", indicator=True)

        turn_count_fail = int((rec["n_turns"].fillna(-1).astype("Int64") !=
                               rec["n_turns_actual"].fillna(-2).astype("Int64")).sum()) if "n_turns" in rec else len(rec)
        student_fail = int((rec["n_student_turns"].fillna(-1).astype("Int64") !=
                            rec["n_student_actual"].fillna(-2).astype("Int64")).sum()) if "n_student_turns" in rec else len(rec)
        tutor_fail = int((rec["n_tutor_turns"].fillna(-1).astype("Int64") !=
                          rec["n_tutor_actual"].fillna(-2).astype("Int64")).sum()) if "n_tutor_turns" in rec else len(rec)
        background_fail = int((rec["n_background_turns"].fillna(-1).astype("Int64") !=
                               rec["n_background_actual"].fillna(-2).astype("Int64")).sum()) if "n_background_turns" in rec else len(rec)
        unknown_fail = int((rec["n_unknown_roles"].fillna(-1).astype("Int64") !=
                            rec["n_other_actual"].fillna(-2).astype("Int64")).sum()) if "n_unknown_roles" in rec else len(rec)
        source_file_fail = int((rec["source_file_relative"].astype("string") !=
                                rec["source_file_relative_actual"].astype("string")).fillna(True).sum()) if "source_file_relative" in rec else len(rec)

        relation_rows += [
            check("3.2", "Session n_turns matches exact streamed turns", turn_count_fail == 0, turn_count_fail, 0),
            check("3.2", "Student turn counts reconcile", student_fail == 0, student_fail, 0),
            check("3.2", "Tutor turn counts reconcile", tutor_fail == 0, tutor_fail, 0),
            check("3.2", "Background turn counts reconcile", background_fail == 0, background_fail, 0),
            check("3.2", "Unknown/other role counts reconcile", unknown_fail == 0, unknown_fail, 0),
            check("3.2", "Session source file matches streamed turns", source_file_fail == 0, source_file_fail, 0),
            check("3.2", "Every session uses exactly one source file",
                  int((turn_session_scan["source_file_nunique"] != 1).sum()) == 0,
                  int((turn_session_scan["source_file_nunique"] != 1).sum()), 0),
        ]

        response_turn_coverage = len(responses_base) - int(
            responses_base["session_id"].astype(str).isin(sessions_without_turns).sum()
        )
        relation_rows.append(check("3.2", "Labelled response→session→turn coverage is 100%",
                                   response_turn_coverage == len(responses_base),
                                   f"{response_turn_coverage}/{len(responses_base)}",
                                   f"{len(responses_base)}/{len(responses_base)}"))

        relation_join_log = pd.DataFrame([
            {
                "join": "responses→sessions", "expected_cardinality": "M:1",
                "left_rows": len(responses_base), "right_rows": len(sessions_candidate),
                "result_rows": response_join_result_rows,
                "unmatched_left_sessions": len(unmatched_response_sessions),
                "unmatched_right_sessions": len(orphan_unlabelled_sessions),
                "duplicate_expansion": max(0, response_join_result_rows - len(responses_base)),
            },
            {
                "join": "sessions→turns", "expected_cardinality": "1:M",
                "left_rows": len(sessions_candidate),
                "right_rows": int(pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows),
                "result_rows": int(turn_session_scan["n_turns_actual"].sum()),
                "unmatched_left_sessions": len(sessions_without_turns),
                "unmatched_right_sessions": len(orphan_turn_sessions),
                "duplicate_expansion": 0,
            }
        ])

    except Exception as e:
        relation_rows.append(check("3.2", "Relational/coverage audit completed", False, type(e).__name__,
                                   "Successful audit", cause=f"{type(e).__name__}: {e}"))
        relation_join_log = pd.DataFrame()

relation_audit = pd.DataFrame(relation_rows)
RELATIONAL_INTEGRITY_READY = bool(len(relation_audit)) and not (
    (~relation_audit["passed"]) & relation_audit["severity"].eq("BLOCKER")
).any()

display(relation_audit)
if len(relation_join_log): display(relation_join_log)
print(f"\nRELATIONAL INTEGRITY READY : {RELATIONAL_INTEGRITY_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.2,Streaming run-boundary self-test,True,PASS,,True,True,Batch grouping must never drop the first row o...
1,3.2,Turn sessions are contiguous blocks,True,PASS,,0,0,
2,3.2,Every response session exists exactly once in ...,True,PASS,,0,0,
3,3.2,Response→Session join preserves response popul...,True,PASS,,35072,35072,Missing or non-unique session key would change...
4,3.2,Every candidate session has at least one turn,True,PASS,,0,0,
5,3.2,No turn belongs to an unknown candidate session,True,PASS,,0,0,
6,3.2,No unused transcript session relative to label...,True,PASS,,0,0,Canonical foundation is expected to match the ...
7,3.2,Session n_turns matches exact streamed turns,True,PASS,,0,0,
8,3.2,Student turn counts reconcile,True,PASS,,0,0,
9,3.2,Tutor turn counts reconcile,True,PASS,,0,0,


,join,expected_cardinality,left_rows,right_rows,result_rows,unmatched_left_sessions,unmatched_right_sessions,duplicate_expansion
0,responses→sessions,M:1,35072,22821,35072,0,0,0
1,sessions→turns,1:M,22821,6139854,6139854,0,0,0



RELATIONAL INTEGRITY READY : True


# 3.3 — Sequence, Ordering, Role & Text Integrity

This section evaluates structural quality without re-implementing the parser.
Exact turn-index/source-row contiguity, first/last flags, relative position, role counts, and source-file consistency come from the independent streamed scan in Section 3.2.
Session-level timestamp/order counters are recomputed from `sessions_candidate` and compared with the certified parser census when that census is present.
Timestamp ties are descriptive information, not corruption.
Raw and normalized text are streamed in batches to derive empty-text counts, `[UNCLEAR]` frequency, normalization changes, maximum length, and optional stored-flag agreement.
Stored optional text flags are checked only when the actual schema contains them.
No row is deleted because of a warning or informational characteristic.
Any drift from the certified parser artifact or broken sequence invariant is blocking.

In [4]:
# ============================================================
# 3.3 — SEQUENCE, ORDERING, ROLE & TEXT INTEGRITY
# ============================================================

structure_rows = []
ordering_summary = pd.DataFrame()
text_quality_summary = pd.DataFrame()

def bool_sum(df, col):
    if col not in df: return None
    return int(df[col].fillna(False).astype(bool).sum())

def numeric_sum(df, col):
    if col not in df: return None
    return int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

def stream_text_quality(path, batch_size=131_072):
    schema_names = set(pq.read_schema(path).names)
    wanted = ["content_raw", "text_norm", "contains_unclear_flag", "empty_content_flag",
              "empty_after_normalization_flag", "role_issue_flag", "ordering_issue_flag",
              "missing_timestamp_flag"]
    cols = [c for c in wanted if c in schema_names]
    stats = {
        "rows": 0, "raw_null": 0, "norm_null": 0, "raw_blank": 0, "norm_blank": 0,
        "unclear_rows_derived": 0, "normalization_changed_rows_derived": 0,
        "max_raw_chars": 0, "max_norm_chars": 0, "raw_chars_ge_1000": 0,
        "stored_unclear_mismatch": 0, "stored_empty_raw_mismatch": 0,
        "stored_empty_norm_mismatch": 0,
    }
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(columns=cols, batch_size=batch_size):
        df = batch.to_pandas()
        n = len(df); stats["rows"] += n

        raw_null = df["content_raw"].isna() if "content_raw" in df else pd.Series([True]*n)
        norm_null = df["text_norm"].isna() if "text_norm" in df else pd.Series([True]*n)
        raw = df["content_raw"].fillna("").astype(str) if "content_raw" in df else pd.Series([""]*n)
        norm = df["text_norm"].fillna("").astype(str) if "text_norm" in df else pd.Series([""]*n)

        raw_blank = raw.str.strip().eq("")
        norm_blank = norm.str.strip().eq("")
        unclear = norm.str.contains("[UNCLEAR]", regex=False)
        changed = raw.ne(norm)
        raw_len = raw.str.len()
        norm_len = norm.str.len()

        stats["raw_null"] += int(raw_null.sum())
        stats["norm_null"] += int(norm_null.sum())
        stats["raw_blank"] += int(raw_blank.sum())
        stats["norm_blank"] += int(norm_blank.sum())
        stats["unclear_rows_derived"] += int(unclear.sum())
        stats["normalization_changed_rows_derived"] += int(changed.sum())
        stats["max_raw_chars"] = max(stats["max_raw_chars"], int(raw_len.max()) if n else 0)
        stats["max_norm_chars"] = max(stats["max_norm_chars"], int(norm_len.max()) if n else 0)
        stats["raw_chars_ge_1000"] += int((raw_len >= 1000).sum())

        if "contains_unclear_flag" in df:
            stats["stored_unclear_mismatch"] += int(
                (df["contains_unclear_flag"].fillna(False).astype(bool).to_numpy() != unclear.to_numpy()).sum()
            )
        if "empty_content_flag" in df:
            stats["stored_empty_raw_mismatch"] += int(
                (df["empty_content_flag"].fillna(False).astype(bool).to_numpy() != raw_blank.to_numpy()).sum()
            )
        if "empty_after_normalization_flag" in df:
            stats["stored_empty_norm_mismatch"] += int(
                (df["empty_after_normalization_flag"].fillna(False).astype(bool).to_numpy() != norm_blank.to_numpy()).sum()
            )
    return stats

if not RELATIONAL_INTEGRITY_READY:
    structure_rows.append(check("3.3", "Section entry gate", False, False, True,
                                cause="Section 3.2 is not ready."))
else:
    try:
        exact_turn_index_fail = int((~turn_session_scan["turn_index_exact"]).sum())
        exact_source_index_fail = int((~turn_session_scan["source_row_index_exact"]).sum())
        first_fail = int((~turn_session_scan["first_flag_exact"]).sum())
        last_fail = int((~turn_session_scan["last_flag_exact"]).sum())
        rel_fail = int((~turn_session_scan["relative_position_valid"]).sum())

        structure_rows += [
            check("3.3", "Within-session turn_index is exact 0..n-1", exact_turn_index_fail == 0,
                  exact_turn_index_fail, 0),
            check("3.3", "Within-session source_row_index is exact 0..n-1", exact_source_index_fail == 0,
                  exact_source_index_fail, 0),
            check("3.3", "Exactly one correct first-turn flag per session", first_fail == 0, first_fail, 0),
            check("3.3", "Exactly one correct last-turn flag per session", last_fail == 0, last_fail, 0),
            check("3.3", "Relative turn positions lie in [0,1]", rel_fail == 0, rel_fail, 0),
        ]

        session_metrics = {
            "timestamp_issue_count": numeric_sum(sessions_candidate, "timestamp_issue_count"),
            "utterance_id_issue_count": numeric_sum(sessions_candidate, "utterance_id_issue_count"),
            "ordering_issue_count": numeric_sum(sessions_candidate, "ordering_issue_count"),
            "unknown_role_count": numeric_sum(sessions_candidate, "unknown_role_count"),
            "empty_content_count": numeric_sum(sessions_candidate, "empty_content_count"),
            "fallback_sessions": bool_sum(sessions_candidate, "fallback_order_used"),
            "ambiguous_sessions": bool_sum(sessions_candidate, "ambiguous_order_flag"),
            "rollover_sessions": int((pd.to_numeric(sessions_candidate["midnight_rollover_count"], errors="coerce").fillna(0) > 0).sum())
                                if "midnight_rollover_count" in sessions_candidate else None,
            "timestamp_id_conflict_sessions": bool_sum(sessions_candidate, "timestamp_id_conflict"),
            "timestamp_source_conflict_sessions": bool_sum(sessions_candidate, "timestamp_source_conflict"),
            "id_source_conflict_sessions": bool_sum(sessions_candidate, "id_source_conflict"),
            "timestamp_tie_groups": numeric_sum(sessions_candidate, "timestamp_tie_count"),
            "timestamp_tie_sessions": int((pd.to_numeric(sessions_candidate["timestamp_tie_count"], errors="coerce").fillna(0) > 0).sum())
                                      if "timestamp_tie_count" in sessions_candidate else None,
            "quality_warning_sessions": int((pd.to_numeric(sessions_candidate["quality_warning_count"], errors="coerce").fillna(0) > 0).sum())
                                        if "quality_warning_count" in sessions_candidate else None,
        }
        ordering_summary = pd.DataFrame([
            {"metric": k, "value": v} for k, v in session_metrics.items()
        ])

        # Compare with parser manifest census only where a corresponding certified value exists.
        census = get_nested(parser_payload, ["production_census"], {}) or {}
        certified_pairs = {
            "fallback_sessions": census.get("fallback_sessions"),
            "ambiguous_sessions": census.get("ambiguous_sessions"),
            "rollover_sessions": census.get("rollover_sessions"),
            "timestamp_tie_groups": census.get("timestamp_tie_groups"),
            "timestamp_tie_sessions": census.get("timestamp_tie_sessions"),
            "quality_warning_sessions": census.get("quality_warning_sessions"),
        }
        for metric, certified in certified_pairs.items():
            if certified is not None and session_metrics.get(metric) is not None:
                structure_rows.append(check("3.3", f"{metric} matches parser census",
                                            int(session_metrics[metric]) == int(certified),
                                            session_metrics[metric], certified,
                                            cause="Candidate session summary drifted from certified parser manifest."))

        # Current corpus is certified conflict-free; any new conflict/ambiguity is a hard drift.
        for metric in ["timestamp_id_conflict_sessions", "timestamp_source_conflict_sessions",
                       "id_source_conflict_sessions", "ambiguous_sessions"]:
            value = session_metrics.get(metric)
            if value is not None:
                structure_rows.append(check("3.3", f"{metric} remains zero", value == 0, value, 0,
                                            cause="Chronology conflict/ambiguity detected in canonical input."))

        # Role census from the exact turn stream.
        role_totals = {
            "student": int(turn_session_scan["n_student_actual"].sum()),
            "tutor": int(turn_session_scan["n_tutor_actual"].sum()),
            "background": int(turn_session_scan["n_background_actual"].sum()),
            "other_or_unknown": int(turn_session_scan["n_other_actual"].sum()),
        }
        prod_roles = {
            "student": census.get("student_turns"),
            "tutor": census.get("tutor_turns"),
            "background": census.get("background_turns"),
            "other_or_unknown": census.get("unknown_turns"),
        }
        for role, actual in role_totals.items():
            certified = prod_roles.get(role)
            if certified is not None:
                structure_rows.append(check("3.3", f"{role} role census matches parser manifest",
                                            actual == int(certified), actual, certified))

        # Full text-quality derivation.
        text_stats = stream_text_quality(TURNS_CANDIDATE_PATH)
        text_quality_summary = pd.DataFrame([{"metric": k, "value": v} for k, v in text_stats.items()])
        structure_rows += [
            check("3.3", "Text-quality scan covers every turn",
                  text_stats["rows"] == int(pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows),
                  text_stats["rows"], int(pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows)),
            check("3.3", "Raw content has no null/blank evidence rows",
                  text_stats["raw_null"] == 0 and text_stats["raw_blank"] == 0,
                  f"null={text_stats['raw_null']}, blank={text_stats['raw_blank']}", 0),
            check("3.3", "Normalized content has no null/blank evidence rows",
                  text_stats["norm_null"] == 0 and text_stats["norm_blank"] == 0,
                  f"null={text_stats['norm_null']}, blank={text_stats['norm_blank']}", 0),
            check("3.3", "Optional stored [UNCLEAR] flag agrees with derived evidence",
                  text_stats["stored_unclear_mismatch"] == 0,
                  text_stats["stored_unclear_mismatch"], 0),
            check("3.3", "Optional stored raw-empty flag agrees with derived evidence",
                  text_stats["stored_empty_raw_mismatch"] == 0,
                  text_stats["stored_empty_raw_mismatch"], 0),
            check("3.3", "Optional stored normalized-empty flag agrees with derived evidence",
                  text_stats["stored_empty_norm_mismatch"] == 0,
                  text_stats["stored_empty_norm_mismatch"], 0),
        ]

        manifest_text = census.get("text_characteristics", {}) if isinstance(census, dict) else {}
        text_compare_map = {
            "unclear_rows": "unclear_rows_derived",
            "normalization_changed_rows": "normalization_changed_rows_derived",
            "empty_raw_content_rows": "raw_blank",
            "empty_normalized_rows": "norm_blank",
        }
        for manifest_key, derived_key in text_compare_map.items():
            if manifest_text.get(manifest_key) is not None:
                structure_rows.append(check("3.3", f"{manifest_key} matches parser census",
                                            int(text_stats[derived_key]) == int(manifest_text[manifest_key]),
                                            text_stats[derived_key], manifest_text[manifest_key]))

        if text_stats["unclear_rows_derived"] > 0:
            add_issue("3.3", "UNCLEAR_MARKER_PRESENT", "INFO",
                      text_stats["unclear_rows_derived"], "Original evidence retained; no deletion.")
        if text_stats["raw_chars_ge_1000"] > 0:
            add_issue("3.3", "VERY_LONG_UTTERANCE", "INFO",
                      text_stats["raw_chars_ge_1000"], "Useful Phase-2 context-length diagnostic; not a Phase-1 failure.")
        consecutive = int(turn_session_scan["consecutive_content_hash_duplicates"].sum())
        if consecutive > 0:
            add_issue("3.3", "CONSECUTIVE_EXACT_CONTENT", "WARNING", consecutive,
                      "Preserved as evidence; inspect during advanced diagnostics.")

    except Exception as e:
        structure_rows.append(check("3.3", "Structural quality audit completed", False, type(e).__name__,
                                    "Successful audit", cause=f"{type(e).__name__}: {e}"))

structure_audit = pd.DataFrame(structure_rows)
STRUCTURAL_INTEGRITY_READY = bool(len(structure_audit)) and not (
    (~structure_audit["passed"]) & structure_audit["severity"].eq("BLOCKER")
).any()

display(structure_audit)
if len(ordering_summary): display(ordering_summary)
if len(text_quality_summary): display(text_quality_summary)
print(f"\nSTRUCTURAL INTEGRITY READY : {STRUCTURAL_INTEGRITY_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.3,Within-session turn_index is exact 0..n-1,True,PASS,,0,0,
1,3.3,Within-session source_row_index is exact 0..n-1,True,PASS,,0,0,
2,3.3,Exactly one correct first-turn flag per session,True,PASS,,0,0,
3,3.3,Exactly one correct last-turn flag per session,True,PASS,,0,0,
4,3.3,"Relative turn positions lie in [0,1]",True,PASS,,0,0,
5,3.3,fallback_sessions matches parser census,True,PASS,,0,0,Candidate session summary drifted from certifi...
6,3.3,ambiguous_sessions matches parser census,True,PASS,,0,0,Candidate session summary drifted from certifi...
7,3.3,rollover_sessions matches parser census,True,PASS,,0,0,Candidate session summary drifted from certifi...
8,3.3,timestamp_tie_groups matches parser census,True,PASS,,325210,325210,Candidate session summary drifted from certifi...
9,3.3,timestamp_tie_sessions matches parser census,True,PASS,,22795,22795,Candidate session summary drifted from certifi...


,metric,value
0,timestamp_issue_count,0
1,utterance_id_issue_count,0
2,ordering_issue_count,0
3,unknown_role_count,0
4,empty_content_count,0
5,fallback_sessions,0
6,ambiguous_sessions,0
7,rollover_sessions,0
8,timestamp_id_conflict_sessions,0
9,timestamp_source_conflict_sessions,0


,metric,value
0,rows,6139854
1,raw_null,0
2,norm_null,0
3,raw_blank,0
4,norm_blank,0
5,unclear_rows_derived,180538
6,normalization_changed_rows_derived,0
7,max_raw_chars,1767
8,max_norm_chars,1767
9,raw_chars_ge_1000,474



STRUCTURAL INTEGRITY READY : True


# 3.4 — Exact Duplicate, Objective Identity & Label-Collision Audit

This section handles only exact deterministic identity relationships; semantic near-duplicate analysis remains Phase 2.
Raw objective identity stays authoritative.
A minimal safe objective representation is derived with NFC normalization, line-ending normalization, and boundary trimming; it is never used to silently merge distinct raw objectives.
`objective_raw` is the authoritative objective identity; `objective_id_raw` is preserved as source metadata and is audited without forcing a one-to-one mapping.
`objective_uid` is a deterministic hash of exact `objective_raw` and is created without any target statistic.
Exact transcript duplicates use the parser’s order-sensitive transcript hashes and are preserved, not deleted.
Repeated `(session_id, objective_raw)` response units are blocking because they violate the prediction-unit contract.
Exact transcript + exact objective groups with conflicting targets are flagged for downstream noise analysis but are not automatically relabelled.
No positive rate, objective prior, or other target aggregate is written to the canonical objective table.

In [5]:
# ============================================================
# 3.4 — EXACT DUPLICATES, OBJECTIVE IDENTITY & LABEL COLLISIONS
# ============================================================

def safe_objective_norm(x):
    if pd.isna(x): return ""
    s = str(x).replace("\r\n", "\n").replace("\r", "\n")
    return unicodedata.normalize("NFC", s).strip()

def objective_uid_from_raw(objective_raw):
    payload = "objective-v2\0" + str(objective_raw)
    return "OBJ_" + hashlib.sha256(payload.encode("utf-8")).hexdigest()

duplicate_rows = []
duplicate_objective_summary = pd.DataFrame()
objective_identity_df = pd.DataFrame()
exact_duplicate_sessions = pd.DataFrame()
label_conflict_groups = pd.DataFrame()

if not STRUCTURAL_INTEGRITY_READY:
    duplicate_rows.append(check("3.4", "Section entry gate", False, False, True,
                                cause="Section 3.3 is not ready."))
else:
    try:
        objective_field = get_nested(data_contract, ["response_schema", "objective_field"], "objective_raw")
        objective_id_field = get_nested(data_contract, ["response_schema", "objective_id_field"], "objective_id_raw")

        needed = ["response_id", "session_id", objective_field, "target"]
        if objective_id_field in responses_base.columns:
            needed.append(objective_id_field)
        work = responses_base[needed].copy()
        work["objective_safe_norm"] = work[objective_field].map(safe_objective_norm)

        raw_objectives = int(work[objective_field].nunique(dropna=False))
        safe_objectives = int(work["objective_safe_norm"].nunique(dropna=False))
        expected_objectives = int(get_nested(
            inventory_manifest, ["objective_reconciliation", "current_authoritative_count"], raw_objectives
        ))

        duplicate_rows.append(check(
            "3.4", "Current raw objective population matches inventory authority",
            raw_objectives == expected_objectives, raw_objectives, expected_objectives,
            cause="Raw exact objective identity is the current authority."
        ))
        duplicate_rows.append(check(
            "3.4", "Safe objective normalization is deterministic/non-empty",
            int(work["objective_safe_norm"].eq("").sum()) == 0,
            int(work["objective_safe_norm"].eq("").sum()), 0
        ))

        safe_collision_groups = int(
            (work.groupby("objective_safe_norm", dropna=False)[objective_field].nunique(dropna=False) > 1).sum()
        )
        if safe_collision_groups:
            add_issue("3.4", "OBJECTIVE_SAFE_NORMALIZATION_COLLISION", "WARNING",
                      safe_collision_groups,
                      "Distinct raw objectives collapse under safe normalization; raw identity is preserved and never merged.")

        objective_ids = None
        id_to_text_max = None
        text_to_id_max = None
        if objective_id_field in work.columns:
            objective_ids = int(work[objective_id_field].nunique(dropna=False))
            id_to_text_max = int(
                work.groupby(objective_id_field, dropna=False)[objective_field].nunique(dropna=False).max()
            )
            text_to_id_max = int(
                work.groupby(objective_field, dropna=False)[objective_id_field].nunique(dropna=False).max()
            )
            if id_to_text_max > 1:
                add_issue("3.4", "OBJECTIVE_ID_MAPS_MULTIPLE_RAW_OBJECTIVES", "WARNING",
                          int((work.groupby(objective_id_field, dropna=False)[objective_field]
                               .nunique(dropna=False) > 1).sum()),
                          "objective_id_raw is source metadata, not canonical identity; raw objective text remains authoritative.")
            if text_to_id_max > 1:
                add_issue("3.4", "RAW_OBJECTIVE_MAPS_MULTIPLE_SOURCE_IDS", "WARNING",
                          int((work.groupby(objective_field, dropna=False)[objective_id_field]
                               .nunique(dropna=False) > 1).sum()),
                          "Raw objective identity is preserved; source-ID inconsistency is documented, not merged.")
            if objective_ids != raw_objectives:
                add_issue("3.4", "OBJECTIVE_ID_POPULATION_DIFFERS_RAW", "INFO",
                          abs(objective_ids - raw_objectives),
                          f"source objective IDs={objective_ids}, raw exact objectives={raw_objectives}; canonical identity uses raw objective text.")

        # Official prediction unit: one row per session × exact raw objective.
        pair_dup_rows = int(work.duplicated(["session_id", objective_field], keep=False).sum())
        duplicate_rows.append(check(
            "3.4", "Session×raw-objective response unit is unique",
            pair_dup_rows == 0, pair_dup_rows, 0,
            cause="Repeated exact (session_id, objective_raw) violates the prediction-unit contract."
        ))

        # One canonical row per exact raw objective. No target-derived statistics.
        base_obj = work[[objective_field, "objective_safe_norm"]].drop_duplicates(
            subset=[objective_field]
        ).copy()
        base_obj["objective_uid"] = base_obj[objective_field].map(objective_uid_from_raw)
        counts = work.groupby(objective_field, dropna=False).agg(
            response_count=("response_id", "size"),
            session_count=("session_id", "nunique")
        ).reset_index()
        objective_identity_df = base_obj.merge(
            counts, on=objective_field, how="left", validate="one_to_one"
        )
        objective_identity_df = objective_identity_df[
            ["objective_uid", objective_field, "objective_safe_norm", "response_count", "session_count"]
        ].sort_values([objective_field], kind="stable").reset_index(drop=True)

        duplicate_rows += [
            check("3.4", "Canonical objective table has one row per raw objective",
                  len(objective_identity_df) == raw_objectives,
                  len(objective_identity_df), raw_objectives),
            check("3.4", "objective_uid is unique",
                  objective_identity_df["objective_uid"].nunique(dropna=False) == len(objective_identity_df),
                  objective_identity_df["objective_uid"].nunique(dropna=False), len(objective_identity_df)),
            check("3.4", "Canonical objective table contains no target statistic",
                  not any(x in c.lower() for c in objective_identity_df.columns
                          for x in ["target","positive_rate","label_mean","objective_prior","target_mean"]),
                  list(objective_identity_df.columns), "No target-derived statistic"),
        ]

        # Exact transcript duplicate groups.
        s = sessions_candidate[
            ["session_id","raw_transcript_hash","normalized_transcript_hash"]
        ].copy()
        raw_sizes = s.groupby("raw_transcript_hash", dropna=False)["session_id"].transform("size")
        norm_sizes = s.groupby("normalized_transcript_hash", dropna=False)["session_id"].transform("size")
        s["exact_duplicate_group"] = np.where(
            raw_sizes.gt(1), "TDUP_" + s["raw_transcript_hash"].astype(str).str[:20], pd.NA
        )
        s["exact_normalized_duplicate_group"] = np.where(
            norm_sizes.gt(1), "NDUP_" + s["normalized_transcript_hash"].astype(str).str[:20], pd.NA
        )
        exact_duplicate_sessions = s

        raw_dup_groups = int(s.loc[raw_sizes.gt(1), "raw_transcript_hash"].nunique())
        raw_dup_sessions = int(raw_sizes.gt(1).sum())
        norm_dup_groups = int(s.loc[norm_sizes.gt(1), "normalized_transcript_hash"].nunique())
        norm_dup_sessions = int(norm_sizes.gt(1).sum())
        if raw_dup_groups:
            add_issue("3.4", "EXACT_CROSS_SESSION_TRANSCRIPT_DUPLICATE", "WARNING",
                      raw_dup_groups, f"{raw_dup_sessions} sessions participate; data preserved.")
        if norm_dup_groups:
            add_issue("3.4", "EXACT_NORMALIZED_TRANSCRIPT_DUPLICATE", "WARNING",
                      norm_dup_groups, f"{norm_dup_sessions} sessions participate; data preserved.")

        # Exact transcript + exact raw objective + conflicting target.
        label_work = work.merge(
            sessions_candidate[["session_id","raw_transcript_hash"]],
            on="session_id", how="left", validate="many_to_one"
        )
        conflict = label_work.groupby(
            ["raw_transcript_hash", objective_field], dropna=False
        ).agg(
            target_nunique=("target","nunique"),
            rows=("response_id","size"),
            sessions=("session_id","nunique")
        ).reset_index()
        label_conflict_groups = conflict[conflict["target_nunique"] > 1].copy()
        if len(label_conflict_groups):
            add_issue("3.4", "LABEL_CONFLICT_CANDIDATE", "WARNING",
                      len(label_conflict_groups),
                      "Same exact transcript + exact raw objective has different targets; never auto-relabel.")

        duplicate_objective_summary = pd.DataFrame([
            {"metric":"raw_objectives_authoritative","value":raw_objectives},
            {"metric":"safe_objectives","value":safe_objectives},
            {"metric":"safe_collision_groups","value":safe_collision_groups},
            {"metric":"source_objective_ids","value":objective_ids},
            {"metric":"max_raw_texts_per_source_id","value":id_to_text_max},
            {"metric":"max_source_ids_per_raw_text","value":text_to_id_max},
            {"metric":"duplicate_session_raw_objective_rows","value":pair_dup_rows},
            {"metric":"raw_duplicate_transcript_groups","value":raw_dup_groups},
            {"metric":"raw_duplicate_sessions","value":raw_dup_sessions},
            {"metric":"normalized_duplicate_groups","value":norm_dup_groups},
            {"metric":"normalized_duplicate_sessions","value":norm_dup_sessions},
            {"metric":"label_conflict_groups","value":len(label_conflict_groups)},
        ])

    except Exception as e:
        duplicate_rows.append(check(
            "3.4", "Duplicate/objective audit completed", False,
            type(e).__name__, "Successful audit",
            cause=f"{type(e).__name__}: {e}"
        ))
        duplicate_objective_summary = pd.DataFrame()

duplicate_objective_audit = pd.DataFrame(duplicate_rows)
DUPLICATE_OBJECTIVE_INTEGRITY_READY = bool(len(duplicate_objective_audit)) and not (
    (~duplicate_objective_audit["passed"]) & duplicate_objective_audit["severity"].eq("BLOCKER")
).any()

display(duplicate_objective_audit)
if len(duplicate_objective_summary):
    display(duplicate_objective_summary)
print(f"\nDUPLICATE / OBJECTIVE INTEGRITY READY : {DUPLICATE_OBJECTIVE_INTEGRITY_READY}")


,section,check,passed,status,severity,observed,expected,cause
0,3.4,Current raw objective population matches inven...,True,PASS,,398,398,Raw exact objective identity is the current au...
1,3.4,Safe objective normalization is deterministic/...,True,PASS,,0,0,
2,3.4,Session×raw-objective response unit is unique,True,PASS,,0,0,"Repeated exact (session_id, objective_raw) vio..."
3,3.4,Canonical objective table has one row per raw ...,True,PASS,,398,398,
4,3.4,objective_uid is unique,True,PASS,,398,398,
5,3.4,Canonical objective table contains no target s...,True,PASS,,"['objective_uid', 'objective_raw', 'objective_...",No target-derived statistic,


,metric,value
0,raw_objectives_authoritative,398
1,safe_objectives,398
2,safe_collision_groups,0
3,source_objective_ids,398
4,max_raw_texts_per_source_id,1
5,max_source_ids_per_raw_text,1
6,duplicate_session_raw_objective_rows,0
7,raw_duplicate_transcript_groups,0
8,raw_duplicate_sessions,0
9,normalized_duplicate_groups,0



DUPLICATE / OBJECTIVE INTEGRITY READY : True


# 3.5 — Frozen Fold & Leakage Integrity

This section certifies the exact frozen validation assignment; no fold is rebuilt or resampled.
The response table and frozen-fold manifest must contain the same response IDs and identical `(response_id, session_id, fold)` assignments.
Every response has exactly one fold and every session must belong to exactly one fold.
The observed fold IDs must equal the frozen fold contract.
Direct cross-fold session overlap is a hard blocker because multiple objectives can share the same transcript.
Different session IDs with an identical transcript hash in different folds are reported separately as exact-duplicate memorization risk; they are not misclassified as direct session leakage.
Fold counts are recorded for later diagnostic stratification.
No target rate by fold or objective is used to alter the foundation.

In [6]:
# ============================================================
# 3.5 — FROZEN FOLD & LEAKAGE INTEGRITY
# ============================================================

fold_rows = []
fold_distribution = pd.DataFrame()
duplicate_cross_fold = pd.DataFrame()

if not DUPLICATE_OBJECTIVE_INTEGRITY_READY:
    fold_rows.append(check("3.5", "Section entry gate", False, False, True,
                           cause="Section 3.4 is not ready."))
else:
    try:
        rf = responses_base[["response_id","session_id","fold"]].copy()
        ff = frozen_folds[["response_id","session_id","fold"]].copy()

        # Compare exact assignments by response ID.
        merged = rf.merge(ff, on="response_id", how="outer", suffixes=("_response","_frozen"),
                          indicator=True, validate="one_to_one")
        unmatched = int((merged["_merge"] != "both").sum())
        session_mismatch = int(
            (merged.loc[merged["_merge"].eq("both"), "session_id_response"].astype(str) !=
             merged.loc[merged["_merge"].eq("both"), "session_id_frozen"].astype(str)).sum()
        )
        fold_mismatch = int(
            (pd.to_numeric(merged.loc[merged["_merge"].eq("both"), "fold_response"], errors="coerce") !=
             pd.to_numeric(merged.loc[merged["_merge"].eq("both"), "fold_frozen"], errors="coerce")).sum()
        )

        response_session_fold_n = rf.groupby("session_id")["fold"].nunique()
        frozen_session_fold_n = ff.groupby("session_id")["fold"].nunique()
        cross_fold_response_sessions = int((response_session_fold_n > 1).sum())
        cross_fold_frozen_sessions = int((frozen_session_fold_n > 1).sum())

        valid_folds = set(get_nested(data_contract, ["fold_contract", "fold_ids"], [0,1,2,3,4]))
        observed_folds = set(pd.to_numeric(ff["fold"], errors="coerce").dropna().astype(int).unique())

        fold_rows += [
            check("3.5", "Response and frozen-fold populations are identical", unmatched == 0,
                  unmatched, 0),
            check("3.5", "Frozen session_id matches response session_id", session_mismatch == 0,
                  session_mismatch, 0),
            check("3.5", "Frozen fold value matches response fold value", fold_mismatch == 0,
                  fold_mismatch, 0),
            check("3.5", "Each response appears once in frozen fold manifest",
                  ff["response_id"].nunique(dropna=False) == len(ff),
                  ff["response_id"].nunique(dropna=False), len(ff)),
            check("3.5", "No session crosses folds in response foundation",
                  cross_fold_response_sessions == 0, cross_fold_response_sessions, 0),
            check("3.5", "No session crosses folds in frozen fold manifest",
                  cross_fold_frozen_sessions == 0, cross_fold_frozen_sessions, 0),
            check("3.5", "Observed fold IDs equal frozen contract", observed_folds == valid_folds,
                  sorted(observed_folds), sorted(valid_folds)),
        ]

        # Exact duplicate transcript risk across different folds.
        session_fold = rf.groupby("session_id", as_index=False)["fold"].first()
        dup = sessions_candidate[["session_id","raw_transcript_hash"]].merge(
            session_fold, on="session_id", how="left", validate="one_to_one"
        )
        dup_stats = dup.groupby("raw_transcript_hash", dropna=False).agg(
            sessions=("session_id","nunique"), folds=("fold","nunique")
        ).reset_index()
        duplicate_cross_fold = dup_stats[(dup_stats["sessions"] > 1) & (dup_stats["folds"] > 1)].copy()
        if len(duplicate_cross_fold):
            add_issue("3.5", "EXACT_DUPLICATE_CROSS_FOLD_RISK", "WARNING",
                      len(duplicate_cross_fold),
                      "Different session IDs share exact transcript bytes across folds; direct session leakage remains zero.")

        fold_distribution = ff.groupby("fold").agg(
            responses=("response_id","size"), sessions=("session_id","nunique")
        ).reset_index().sort_values("fold")

    except Exception as e:
        fold_rows.append(check("3.5", "Frozen fold/leakage audit completed", False, type(e).__name__,
                               "Successful audit", cause=f"{type(e).__name__}: {e}"))

fold_audit = pd.DataFrame(fold_rows)
FOLD_LEAKAGE_INTEGRITY_READY = bool(len(fold_audit)) and not (
    (~fold_audit["passed"]) & fold_audit["severity"].eq("BLOCKER")
).any()

display(fold_audit)
if len(fold_distribution): display(fold_distribution)
if len(duplicate_cross_fold):
    print("\nExact duplicate cross-fold risk groups (warning, not direct session leakage):")
    display(duplicate_cross_fold.head(20))
print(f"\nFOLD / LEAKAGE INTEGRITY READY : {FOLD_LEAKAGE_INTEGRITY_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.5,Response and frozen-fold populations are ident...,True,PASS,,0,0,
1,3.5,Frozen session_id matches response session_id,True,PASS,,0,0,
2,3.5,Frozen fold value matches response fold value,True,PASS,,0,0,
3,3.5,Each response appears once in frozen fold mani...,True,PASS,,35072,35072,
4,3.5,No session crosses folds in response foundation,True,PASS,,0,0,
5,3.5,No session crosses folds in frozen fold manifest,True,PASS,,0,0,
6,3.5,Observed fold IDs equal frozen contract,True,PASS,,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[0, 1, 2, 3, 4]",


,fold,responses,sessions
0,0,6980,4590
1,1,7018,4569
2,2,7019,4570
3,3,7011,4541
4,4,7044,4551



FOLD / LEAKAGE INTEGRITY READY : True


# 3.6 — Raw Traceability & Training/Inference Symmetry

This section independently verifies that representative candidate turns can be reconstructed from their registered raw transcript source by `source_file_relative` and `source_row_index`.
The sample is deterministic and covers short, long, high-tie, background-heavy, long-duration, and stable control sessions where available.
Raw CSV fields are compared exactly against the preserved parser fields; source file SHA256 is also checked against the candidate session record.
This is a physical provenance test, not a semantic review.
The turn/session schemas are then scanned for prohibited target, fold, objective, prior, OOF, prediction, or validation-derived columns.
The parser manifest must remain explicitly label-blind.
Response tables may contain `target` and `fold`; evidence tables may not.
A failure here means the candidate evidence layer is not safe for canonical publication.

In [7]:
# ============================================================
# 3.6 — RAW TRACEABILITY & TRAINING/INFERENCE SYMMETRY
# ============================================================

def stable_rank_key(x):
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()

def resolve_raw_source_file(relative_value):
    rel_text = str(relative_value).replace("\\", os.sep).replace("/", os.sep)
    rel = Path(rel_text)
    candidates = []
    if TRANSCRIPT_ROOT is not None:
        candidates += [TRANSCRIPT_ROOT / rel, TRANSCRIPT_ROOT / rel.name, TRANSCRIPT_ROOT.parent / rel]
    if DATA_ROOT is not None:
        candidates += [DATA_ROOT / rel, DATA_ROOT / rel.name]
    if PROJECT_ROOT is not None:
        candidates += [PROJECT_ROOT / rel]
    seen = set()
    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        key = str(rp)
        if key not in seen:
            seen.add(key)
            if p.exists() and p.is_file():
                return p
    return None

def raw_csv_records(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return list(reader), list(reader.fieldnames or [])

trace_rows = []
traceability_detail = pd.DataFrame()
symmetry_detail = pd.DataFrame()

if not FOLD_LEAKAGE_INTEGRITY_READY:
    trace_rows.append(check("3.6", "Section entry gate", False, False, True,
                            cause="Section 3.5 is not ready."))
else:
    try:
        s = sessions_candidate.copy()
        nturns_num = pd.to_numeric(s["n_turns"], errors="coerce").fillna(0)
        if "n_background_turns" in s:
            background_num = pd.to_numeric(s["n_background_turns"], errors="coerce").fillna(0)
        else:
            background_num = pd.Series(0, index=s.index, dtype="float64")
        s["_background_prop"] = np.where(
            nturns_num > 0, background_num / nturns_num.replace(0, np.nan), 0
        )
        selected = []
        def choose(idx, reason):
            if idx is None: return
            sid = str(s.loc[idx, "session_id"])
            if sid not in {x[0] for x in selected}: selected.append((sid, reason))

        if len(s):
            choose(s["n_turns"].astype(int).idxmin(), "shortest")
            choose(s["n_turns"].astype(int).idxmax(), "longest")
            if "duration_seconds" in s and s["duration_seconds"].notna().any():
                choose(pd.to_numeric(s["duration_seconds"], errors="coerce").idxmax(), "max_duration")
            if "timestamp_tie_count" in s:
                choose(pd.to_numeric(s["timestamp_tie_count"], errors="coerce").fillna(0).idxmax(), "max_timestamp_ties")
            choose(s["_background_prop"].idxmax(), "background_heavy")

            for sid in sorted(s["session_id"].astype(str), key=stable_rank_key):
                if len(selected) >= 12: break
                if sid not in {x[0] for x in selected}: selected.append((sid, "stable_control"))

        sample_ids = [x[0] for x in selected]
        sample_reason = dict(selected)

        # Read only selected turns.
        dataset = ds.dataset(TURNS_CANDIDATE_PATH, format="parquet")
        available = set(dataset.schema.names)
        raw_cols = ["session_id","source_file_relative","source_row_index","session_id_raw",
                    "utterance_id_raw","role_raw","content_raw","timestamp_raw","turn_uid"]
        raw_cols = [c for c in raw_cols if c in available]
        sample_turns = dataset.to_table(
            columns=raw_cols, filter=ds.field("session_id").isin(sample_ids)
        ).to_pandas()

        raw_required = set(get_nested(data_contract, ["transcript_schema","required_fields"],
                                      ["session_id","utterance_id","role","content","timestamp"]))
        detail = []
        session_lookup = sessions_candidate.set_index(sessions_candidate["session_id"].astype(str), drop=False)

        field_pairs = [
            ("session_id_raw","session_id"),
            ("utterance_id_raw","utterance_id"),
            ("role_raw","role"),
            ("content_raw","content"),
            ("timestamp_raw","timestamp"),
        ]

        for sid in sample_ids:
            cause = []
            g = sample_turns[sample_turns["session_id"].astype(str).eq(sid)].copy()
            if g.empty:
                detail.append({"session_id":sid,"reason":sample_reason[sid],"passed":False,
                               "cause":"Selected session not found in candidate turns."})
                continue

            rel = str(g["source_file_relative"].iloc[0])
            raw_path = resolve_raw_source_file(rel)
            if raw_path is None:
                detail.append({"session_id":sid,"reason":sample_reason[sid],"passed":False,
                               "cause":f"Raw source file cannot be resolved: {rel}"})
                continue

            try:
                rows, header = raw_csv_records(raw_path)
                missing_header = sorted(raw_required - set(header))
                if missing_header:
                    cause.append(f"missing raw CSV fields={missing_header}")

                if len(rows) != len(g):
                    cause.append(f"row_count raw={len(rows)} candidate={len(g)}")

                by_source = g.set_index(pd.to_numeric(g["source_row_index"], errors="coerce").astype("Int64"))
                for i, rr in enumerate(rows):
                    if i not in by_source.index:
                        cause.append(f"source_row_index {i} missing")
                        continue
                    cr = by_source.loc[i]
                    if isinstance(cr, pd.DataFrame):
                        cause.append(f"duplicate source_row_index {i}")
                        continue
                    for cand_col, raw_col in field_pairs:
                        if cand_col in cr.index and raw_col in rr:
                            cv = "" if pd.isna(cr[cand_col]) else str(cr[cand_col])
                            rv = "" if rr[raw_col] is None else str(rr[raw_col])
                            if cv != rv:
                                cause.append(f"{cand_col}!={raw_col} at row {i}")
                                break

                if sid in session_lookup.index and "file_sha256" in session_lookup.columns:
                    sr = session_lookup.loc[sid]
                    if isinstance(sr, pd.DataFrame): sr = sr.iloc[0]
                    source_hash = sha256_file(raw_path)
                    if str(sr["file_sha256"]) != source_hash:
                        cause.append("raw file SHA256 mismatch")

            except Exception as e:
                cause.append(f"{type(e).__name__}: {e}")

            detail.append({
                "session_id":sid, "reason":sample_reason[sid],
                "candidate_turns":len(g), "raw_file":str(raw_path),
                "passed":len(cause)==0, "cause":" | ".join(cause)
            })

        traceability_detail = pd.DataFrame(detail)
        trace_failures = int((~traceability_detail["passed"]).sum()) if len(traceability_detail) else len(sample_ids)

        trace_rows += [
            check("3.6", "Deterministic raw-traceability sample selected", len(sample_ids) >= 10,
                  len(sample_ids), ">=10"),
            check("3.6", "Raw→candidate reconstruction sample passes exactly",
                  trace_failures == 0 and len(traceability_detail) == len(sample_ids),
                  f"failures={trace_failures}, reviewed={len(traceability_detail)}", f"0/{len(sample_ids)} failures"),
        ]

        # Inference symmetry: evidence tables cannot contain target/objective/fold/model-derived fields.
        turn_cols = list(pq.read_schema(TURNS_CANDIDATE_PATH).names)
        session_cols = list(pq.read_schema(SESSIONS_CANDIDATE_PATH).names)
        forbidden_terms = set(str(x).lower() for x in get_nested(
            data_contract, ["parser_restrictions","forbidden_inputs"], []
        ))
        forbidden_terms |= {
            "target","label","fold","objective","objective_prior","positive_rate","target_mean",
            "oof","prediction","validation_metric","baseline_prediction","test_population_statistics"
        }

        def prohibited(cols):
            hits = []
            for c in cols:
                lc = c.lower()
                matched = sorted(t for t in forbidden_terms if t and t in lc)
                if matched: hits.append({"column":c,"matched_terms":"|".join(matched)})
            return hits

        turn_hits = prohibited(turn_cols)
        session_hits = prohibited(session_cols)
        parser_label_blind = bool(
            get_nested(parser_payload, ["parser_policy","label_blind"],
                       get_nested(data_contract, ["parser_restrictions","label_blind"], False))
        )

        symmetry_detail = pd.DataFrame(
            [{"table":"turns_candidate", **x} for x in turn_hits] +
            [{"table":"sessions_candidate", **x} for x in session_hits]
        )
        trace_rows += [
            check("3.6", "Turn evidence schema has no prohibited target/model inputs",
                  len(turn_hits) == 0, [x["column"] for x in turn_hits], []),
            check("3.6", "Session evidence schema has no prohibited target/model inputs",
                  len(session_hits) == 0, [x["column"] for x in session_hits], []),
            check("3.6", "Parser remains explicitly label-blind", parser_label_blind,
                  parser_label_blind, True),
        ]

    except Exception as e:
        trace_rows.append(check("3.6", "Traceability/symmetry audit completed", False, type(e).__name__,
                                "Successful audit", cause=f"{type(e).__name__}: {e}"))

traceability_audit = pd.DataFrame(trace_rows)
TRACEABILITY_SYMMETRY_READY = bool(len(traceability_audit)) and not (
    (~traceability_audit["passed"]) & traceability_audit["severity"].eq("BLOCKER")
).any()

display(traceability_audit)
if len(traceability_detail): display(traceability_detail)
if len(symmetry_detail):
    print("\nProhibited evidence-schema fields:")
    display(symmetry_detail)
print(f"\nTRACEABILITY / SYMMETRY READY : {TRACEABILITY_SYMMETRY_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.6,Deterministic raw-traceability sample selected,True,PASS,,12,>=10,
1,3.6,Raw→candidate reconstruction sample passes exa...,True,PASS,,"failures=0, reviewed=12",0/12 failures,
2,3.6,Turn evidence schema has no prohibited target/...,True,PASS,,[],[],
3,3.6,Session evidence schema has no prohibited targ...,True,PASS,,[],[],
4,3.6,Parser remains explicitly label-blind,True,PASS,,True,True,


,session_id,reason,candidate_turns,raw_file,passed,cause
0,jlntsbf,shortest,15,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
1,bvnewyc,longest,622,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
2,eafvzsi,max_duration,471,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
3,egiejia,max_timestamp_ties,504,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
4,mnvgsri,background_heavy,64,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
5,exrphmf,stable_control,235,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
6,dcxwnrr,stable_control,205,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
7,kegxjly,stable_control,195,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
8,jenrefs,stable_control,230,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,
9,cwpovtt,stable_control,252,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,True,



TRACEABILITY / SYMMETRY READY : True


# 3.7 — Canonical Table Construction

Canonical construction is permitted only after Sections 3.0–3.6 pass all blocking gates.
`responses.parquet` preserves the frozen response population and adds only deterministic `objective_uid`.
`turns.parquet` is intentionally evidence-identical to the certified candidate turn artifact; no semantic or model column is added.
`sessions.parquet` preserves candidate session fields and adds only target-free exact transcript duplicate group identifiers.
`objectives.parquet` contains exactly one deterministic identity per exact raw objective, with safe text, response count, and session count.
Objective counts are descriptive population counts, not target statistics.
No positive rate, target mean, prior, OOF signal, or retrieval feature enters any canonical evidence table.
The construction remains in memory or as a source reference until the safe publication section verifies all contracts.

In [8]:
# ============================================================
# 3.7 — CANONICAL TABLE CONSTRUCTION
# ============================================================

canonical_build_rows = []
responses_canonical = pd.DataFrame()
sessions_canonical = pd.DataFrame()
objectives_canonical = pd.DataFrame()
turns_canonical_source = None

UPSTREAM_INTEGRITY_READY = all([
    INTEGRITY_BOOTSTRAP_READY, ENTITY_KEY_INTEGRITY_READY, RELATIONAL_INTEGRITY_READY,
    STRUCTURAL_INTEGRITY_READY, DUPLICATE_OBJECTIVE_INTEGRITY_READY,
    FOLD_LEAKAGE_INTEGRITY_READY, TRACEABILITY_SYMMETRY_READY
])

if not UPSTREAM_INTEGRITY_READY:
    canonical_build_rows.append(check("3.7", "All blocking integrity gates pass", False,
                                      [INTEGRITY_BOOTSTRAP_READY, ENTITY_KEY_INTEGRITY_READY,
                                       RELATIONAL_INTEGRITY_READY, STRUCTURAL_INTEGRITY_READY,
                                       DUPLICATE_OBJECTIVE_INTEGRITY_READY,
                                       FOLD_LEAKAGE_INTEGRITY_READY, TRACEABILITY_SYMMETRY_READY],
                                      "All True", cause="Canonical construction disabled."))
else:
    try:
        objective_field = get_nested(data_contract, ["response_schema", "objective_field"], "objective_raw")

        # Objective table is already target-free and has one row per exact raw objective.
        objectives_canonical = objective_identity_df.copy()

        # Response table preserves all frozen response columns and adds deterministic objective_uid.
        obj_map = objectives_canonical[[objective_field, "objective_uid"]]
        responses_canonical = responses_base.merge(
            obj_map, on=[objective_field], how="left", validate="many_to_one"
        )
        if "objective_uid" in responses_canonical:
            cols = list(responses_canonical.columns)
            cols.remove("objective_uid")
            insert_at = cols.index(objective_field) + 1 if objective_field in cols else len(cols)
            cols.insert(insert_at, "objective_uid")
            responses_canonical = responses_canonical[cols]

        # Session table adds only target-free exact duplicate identifiers.
        sessions_canonical = sessions_candidate.merge(
            exact_duplicate_sessions[["session_id","exact_duplicate_group","exact_normalized_duplicate_group"]],
            on="session_id", how="left", validate="one_to_one"
        )

        # Turn table must remain byte/evidence identical; publication copies the certified Parquet.
        turns_canonical_source = TURNS_CANDIDATE_PATH

        canonical_build_rows += [
            check("3.7", "Canonical responses preserve row population",
                  len(responses_canonical) == len(responses_base),
                  len(responses_canonical), len(responses_base)),
            check("3.7", "Every canonical response has objective_uid",
                  "objective_uid" in responses_canonical and int(responses_canonical["objective_uid"].isna().sum()) == 0,
                  int(responses_canonical["objective_uid"].isna().sum()) if "objective_uid" in responses_canonical else "missing", 0),
            check("3.7", "Canonical response_id remains unique",
                  responses_canonical["response_id"].nunique(dropna=False) == len(responses_canonical),
                  responses_canonical["response_id"].nunique(dropna=False), len(responses_canonical)),
            check("3.7", "Canonical sessions preserve session population",
                  len(sessions_canonical) == len(sessions_candidate),
                  len(sessions_canonical), len(sessions_candidate)),
            check("3.7", "Canonical session_id remains unique",
                  sessions_canonical["session_id"].nunique(dropna=False) == len(sessions_canonical),
                  sessions_canonical["session_id"].nunique(dropna=False), len(sessions_canonical)),
            check("3.7", "Canonical objective population is authoritative",
                  len(objectives_canonical) == int(get_nested(
                      inventory_manifest, ["objective_reconciliation","current_authoritative_count"],
                      len(objectives_canonical))),
                  len(objectives_canonical),
                  get_nested(inventory_manifest, ["objective_reconciliation","current_authoritative_count"])),
            check("3.7", "Canonical objective table remains target-free",
                  not any(term in c.lower() for c in objectives_canonical.columns
                          for term in ["target","positive_rate","label_mean","objective_prior","target_mean"]),
                  list(objectives_canonical.columns), "No target-derived statistic"),
            check("3.7", "Canonical turns source is certified candidate artifact",
                  Path(turns_canonical_source) == Path(TURNS_CANDIDATE_PATH),
                  turns_canonical_source, TURNS_CANDIDATE_PATH),
        ]

    except Exception as e:
        canonical_build_rows.append(check("3.7", "Canonical construction completed", False, type(e).__name__,
                                          "Successful construction", cause=f"{type(e).__name__}: {e}"))

canonical_build_audit = pd.DataFrame(canonical_build_rows)
CANONICAL_BUILD_READY = bool(len(canonical_build_audit)) and not (
    (~canonical_build_audit["passed"]) & canonical_build_audit["severity"].eq("BLOCKER")
).any()

display(canonical_build_audit)
print(f"\nCANONICAL BUILD READY : {CANONICAL_BUILD_READY}")

,section,check,passed,status,severity,observed,expected,cause
0,3.7,Canonical responses preserve row population,True,PASS,,35072,35072,
1,3.7,Every canonical response has objective_uid,True,PASS,,0,0,
2,3.7,Canonical response_id remains unique,True,PASS,,35072,35072,
3,3.7,Canonical sessions preserve session population,True,PASS,,22821,22821,
4,3.7,Canonical session_id remains unique,True,PASS,,22821,22821,
5,3.7,Canonical objective population is authoritative,True,PASS,,398,398,
6,3.7,Canonical objective table remains target-free,True,PASS,,"['objective_uid', 'objective_raw', 'objective_...",No target-derived statistic,
7,3.7,Canonical turns source is certified candidate ...,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,



CANONICAL BUILD READY : True


# 3.8 — Safe Canonical Publication & Serialization Audit

Publication is a gated filesystem transaction.
All four canonical artifacts are first created inside a staging directory under `03_integrity`.
The large turn artifact is copied byte-for-byte from the certified candidate file, guaranteeing that canonical publication cannot rewrite transcript evidence.
The three small tables are written as Parquet, reloaded, and checked for row count, schema/key integrity, and content identity.
The staged turn file must match the candidate SHA256 exactly.
Only after every staged artifact verifies is the canonical directory promoted.
If a previous canonical directory exists, it is temporarily renamed and automatically restored if promotion fails.
Final files are hashed again after promotion.
Any staging, serialization, or promotion mismatch leaves `CANONICAL_PUBLICATION_READY=False`.

In [9]:
# ============================================================
# 3.8 — SAFE CANONICAL PUBLICATION & SERIALIZATION AUDIT
# ============================================================

def dataframe_digest(df):
    # Deterministic digest for small canonical tables, including column order and values.
    h = hashlib.sha256()
    h.update(("\x1f".join(map(str, df.columns)) + "\n").encode("utf-8"))
    for row in df.itertuples(index=False, name=None):
        h.update(("\x1e".join("" if pd.isna(v) else str(v) for v in row) + "\n").encode("utf-8"))
    return h.hexdigest()

def write_df_parquet(df, path):
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, path, compression="zstd", use_dictionary=True)

def safe_promote_directory(staging, final_dir):
    staging, final_dir = Path(staging), Path(final_dir)
    backup = final_dir.with_name("." + final_dir.name + ".backup_" + INTEGRITY_RUN_ID)
    try:
        if backup.exists(): shutil.rmtree(backup)
        if final_dir.exists():
            os.replace(final_dir, backup)
        os.replace(staging, final_dir)
        if backup.exists(): shutil.rmtree(backup)
        return True, ""
    except Exception as e:
        try:
            if final_dir.exists() and backup.exists():
                shutil.rmtree(final_dir)
                os.replace(backup, final_dir)
            elif (not final_dir.exists()) and backup.exists():
                os.replace(backup, final_dir)
        except Exception:
            pass
        return False, f"{type(e).__name__}: {e}"

publication_rows = []
canonical_artifacts = pd.DataFrame()
CANONICAL_PUBLICATION_READY = False
publication_cause = ""

if not CANONICAL_BUILD_READY:
    publication_rows.append(check("3.8", "Canonical build gate", False, False, True,
                                  cause="Section 3.7 is not ready; nothing was written."))
else:
    staging = INTEGRITY_DIR / (".canonical_staging_" + INTEGRITY_RUN_ID)
    try:
        INTEGRITY_DIR.mkdir(parents=True, exist_ok=True)
        if staging.exists(): shutil.rmtree(staging)
        staging.mkdir(parents=True)

        staged_paths = {
            "responses": staging / "responses.parquet",
            "turns": staging / "turns.parquet",
            "sessions": staging / "sessions.parquet",
            "objectives": staging / "objectives.parquet",
        }

        write_df_parquet(responses_canonical, staged_paths["responses"])
        shutil.copy2(TURNS_CANDIDATE_PATH, staged_paths["turns"])
        write_df_parquet(sessions_canonical, staged_paths["sessions"])
        write_df_parquet(objectives_canonical, staged_paths["objectives"])
        gc.collect()

        # Reload small artifacts.
        rr = pq.read_table(staged_paths["responses"]).to_pandas()
        ss = pq.read_table(staged_paths["sessions"]).to_pandas()
        oo = pq.read_table(staged_paths["objectives"]).to_pandas()

        candidate_turn_hash = sha256_file(TURNS_CANDIDATE_PATH)
        staged_turn_hash = sha256_file(staged_paths["turns"])
        turn_schema_equal = pq.read_schema(staged_paths["turns"]) == pq.read_schema(TURNS_CANDIDATE_PATH)
        turn_rows_equal = int(pq.read_metadata(staged_paths["turns"]).num_rows) == int(
            pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows
        )

        stage_checks = [
            check("3.8", "Staged responses reload exactly",
                  len(rr) == len(responses_canonical)
                  and rr["response_id"].nunique(dropna=False) == len(rr)
                  and dataframe_digest(rr) == dataframe_digest(responses_canonical),
                  len(rr), len(responses_canonical)),
            check("3.8", "Staged turns are byte-identical to certified candidate",
                  staged_turn_hash == candidate_turn_hash and turn_schema_equal and turn_rows_equal,
                  staged_turn_hash, candidate_turn_hash),
            check("3.8", "Staged sessions reload exactly",
                  len(ss) == len(sessions_canonical)
                  and ss["session_id"].nunique(dropna=False) == len(ss)
                  and dataframe_digest(ss) == dataframe_digest(sessions_canonical),
                  len(ss), len(sessions_canonical)),
            check("3.8", "Staged objectives reload exactly",
                  len(oo) == len(objectives_canonical)
                  and oo["objective_uid"].nunique(dropna=False) == len(oo)
                  and dataframe_digest(oo) == dataframe_digest(objectives_canonical),
                  len(oo), len(objectives_canonical)),
        ]
        publication_rows.extend(stage_checks)

        stage_ready = all(x["passed"] for x in stage_checks)
        # Release any Arrow/Pandas-backed references before Windows directory promotion.
        del rr, ss, oo
        gc.collect()
        if stage_ready:
            promoted, promote_error = safe_promote_directory(staging, CANONICAL_DIR)
            publication_rows.append(check("3.8", "Canonical directory promotion completed",
                                          promoted, CANONICAL_DIR, "Promoted directory",
                                          cause=promote_error))
            if promoted:
                final_records = []
                for name in ["responses","turns","sessions","objectives"]:
                    p = CANONICAL_DIR / f"{name}.parquet"
                    final_records.append({
                        "artifact": name,
                        "relative_path": p.relative_to(FOUNDATION_ROOT).as_posix(),
                        "rows": int(pq.read_metadata(p).num_rows),
                        "schema_fields": len(pq.read_schema(p)),
                        "sha256": sha256_file(p),
                    })
                canonical_artifacts = pd.DataFrame(final_records)

                final_turn_hash = canonical_artifacts.loc[
                    canonical_artifacts["artifact"].eq("turns"), "sha256"
                ].iloc[0]
                final_checks = [
                    check("3.8", "Published responses row count stable",
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("responses"),"rows"].iloc[0])
                          == len(responses_canonical),
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("responses"),"rows"].iloc[0]),
                          len(responses_canonical)),
                    check("3.8", "Published turns remain candidate-identical",
                          final_turn_hash == candidate_turn_hash, final_turn_hash, candidate_turn_hash),
                    check("3.8", "Published sessions row count stable",
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("sessions"),"rows"].iloc[0])
                          == len(sessions_canonical),
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("sessions"),"rows"].iloc[0]),
                          len(sessions_canonical)),
                    check("3.8", "Published objectives row count stable",
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("objectives"),"rows"].iloc[0])
                          == len(objectives_canonical),
                          int(canonical_artifacts.loc[canonical_artifacts["artifact"].eq("objectives"),"rows"].iloc[0]),
                          len(objectives_canonical)),
                ]
                publication_rows.extend(final_checks)
        else:
            publication_cause = "One or more staged serialization checks failed."
            shutil.rmtree(staging, ignore_errors=True)

    except Exception as e:
        publication_cause = f"{type(e).__name__}: {e}"
        publication_rows.append(check("3.8", "Canonical staging/publication completed", False,
                                      type(e).__name__, "Successful publication", cause=publication_cause))
        try:
            if staging.exists(): shutil.rmtree(staging, ignore_errors=True)
        except Exception:
            pass

publication_audit = pd.DataFrame(publication_rows)
CANONICAL_PUBLICATION_READY = bool(len(publication_audit)) and not (
    (~publication_audit["passed"]) & publication_audit["severity"].eq("BLOCKER")
).any() and CANONICAL_DIR is not None and CANONICAL_DIR.exists() and len(canonical_artifacts) == 4

display(publication_audit)
if len(canonical_artifacts): display(canonical_artifacts)
print(f"\nCANONICAL PUBLICATION READY : {CANONICAL_PUBLICATION_READY}")
if publication_cause:
    print(f"Publication cause: {publication_cause}")

,section,check,passed,status,severity,observed,expected,cause
0,3.8,Staged responses reload exactly,True,PASS,,35072,35072,
1,3.8,Staged turns are byte-identical to certified c...,True,PASS,,8c9c6103b655f4539e5869fc104eb9ce11af38554f8f95...,8c9c6103b655f4539e5869fc104eb9ce11af38554f8f95...,
2,3.8,Staged sessions reload exactly,True,PASS,,22821,22821,
3,3.8,Staged objectives reload exactly,True,PASS,,398,398,
4,3.8,Canonical directory promotion completed,True,PASS,,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...,Promoted directory,
5,3.8,Published responses row count stable,True,PASS,,35072,35072,
6,3.8,Published turns remain candidate-identical,True,PASS,,8c9c6103b655f4539e5869fc104eb9ce11af38554f8f95...,8c9c6103b655f4539e5869fc104eb9ce11af38554f8f95...,
7,3.8,Published sessions row count stable,True,PASS,,22821,22821,
8,3.8,Published objectives row count stable,True,PASS,,398,398,


,artifact,relative_path,rows,schema_fields,sha256
0,responses,03_integrity/canonical/responses.parquet,35072,7,de1e617ac8b955c6d7af0c8d41cb09dd7b5f754cbd12e9...
1,turns,03_integrity/canonical/turns.parquet,6139854,52,8c9c6103b655f4539e5869fc104eb9ce11af38554f8f95...
2,sessions,03_integrity/canonical/sessions.parquet,22821,33,ebfb690f653e3d96531469d245df8e3b74d9b12bdc165b...
3,objectives,03_integrity/canonical/objectives.parquet,398,5,fb89e1e1ac17ac664d0ca22811e4e58827a55cdddbd9e7...



CANONICAL PUBLICATION READY : True


# 3.9 — Phase Manifest, Audit Workbook & Final Hard Gate

The final section freezes the evidence produced by all prior integrity checks.
Seventeen non-negotiable gates map the Phase-1 specification to machine-readable PASS/FAIL decisions.
The audit workbook contains one concise sheet per integrity responsibility rather than many small files.
`phase1_manifest.json` records source bindings, parser lineage, canonical artifact hashes, population census, objective identity status, duplicate warnings, and section gates.
`phase1_gate.json` records the final hard-gate decision.
The report and JSON files use safe temporary-write/read-back promotion.
`PHASE_1_FOUNDATION_READY=True` is emitted only when all 17 hard gates pass, canonical publication succeeds, and the mandatory audit/manifest/gate artifacts are successfully written.
Warnings such as exact cross-session duplicates or `[UNCLEAR]` remain visible but do not silently block a scientifically valid foundation.
After a full PASS, Phase 1 is frozen and the canonical tables are ready for architecture-driving diagnostics and objective-conditioned retrieval.

In [10]:
# ============================================================
# 3.9 — PHASE MANIFEST, AUDIT WORKBOOK & FINAL HARD GATE
# ============================================================

def excel_safe_df(df):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame()
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].map(
                lambda x: json.dumps(json_safe(x), ensure_ascii=False, sort_keys=True)
                if isinstance(x, (dict, list, tuple, set)) else x
            )
    return out

# Explicit conditions used by the 17 hard gates.
def passed_check(df, contains):
    if df is None or df.empty: return False
    m = df["check"].astype(str).str.contains(contains, regex=False)
    return bool(m.any() and df.loc[m, "passed"].all())

response_id_unique_ok = passed_check(entity_key_audit, "response_id globally unique")
target_valid_ok = passed_check(entity_key_audit, "Target is binary")
objective_population_ok = (
    DUPLICATE_OBJECTIVE_INTEGRITY_READY
    and len(objectives_canonical) == int(get_nested(
        inventory_manifest, ["objective_reconciliation","current_authoritative_count"],
        len(objectives_canonical)
    ))
)
coverage_ok = passed_check(relation_audit, "coverage is 100%")
fold_assignment_ok = (
    passed_check(fold_audit, "Response and frozen-fold populations are identical")
    and passed_check(fold_audit, "Frozen fold value matches response fold value")
)
zero_cross_fold_ok = (
    passed_check(fold_audit, "No session crosses folds in response foundation")
    and passed_check(fold_audit, "No session crosses folds in frozen fold manifest")
)
parser_deterministic_ok = bool(get_nested(
    parser_payload, ["certification","determinism_reconstruction_ready"], False
))
turn_identity_ok = (
    passed_check(entity_key_audit, "turn_uid globally unique")
    and passed_check(entity_key_audit, "source_row_uid globally unique")
)
sequence_ok = passed_check(structure_audit, "Within-session turn_index is exact 0..n-1")
relations_ok = RELATIONAL_INTEGRITY_READY
no_join_expansion_ok = True
if "relation_join_log" in globals() and isinstance(relation_join_log, pd.DataFrame) and len(relation_join_log):
    no_join_expansion_ok = bool((pd.to_numeric(relation_join_log["duplicate_expansion"], errors="coerce").fillna(0) == 0).all())
traceability_ok = passed_check(traceability_audit, "Raw→candidate reconstruction sample passes exactly")
serialization_ok = CANONICAL_PUBLICATION_READY
inference_symmetry_ok = TRACEABILITY_SYMMETRY_READY
source_identity_ok = INTEGRITY_BOOTSTRAP_READY
population_alignment_ok = (
    len(responses_base) == int(get_nested(data_contract, ["source_binding","expected_response_count"], len(responses_base)))
    and ENTITY_KEY_INTEGRITY_READY
)
publication_after_pass_ok = CANONICAL_PUBLICATION_READY and UPSTREAM_INTEGRITY_READY

final_gate_records = [
    {"gate":"G01","requirement":"Current source/schema identity valid","passed":source_identity_ok},
    {"gate":"G02","requirement":"Feature-label response population aligned","passed":population_alignment_ok},
    {"gate":"G03","requirement":"response_id unique","passed":response_id_unique_ok},
    {"gate":"G04","requirement":"Target valid","passed":target_valid_ok},
    {"gate":"G05","requirement":"Current authoritative objective population deterministic","passed":objective_population_ok},
    {"gate":"G06","requirement":"Labelled-session transcript coverage valid","passed":coverage_ok},
    {"gate":"G07","requirement":"Every response assigned to exactly one frozen fold","passed":fold_assignment_ok},
    {"gate":"G08","requirement":"Zero session cross-fold overlap","passed":zero_cross_fold_ok},
    {"gate":"G09","requirement":"Parser deterministic","passed":parser_deterministic_ok},
    {"gate":"G10","requirement":"Turn identity unique","passed":turn_identity_ok},
    {"gate":"G11","requirement":"Within-session turn sequence contiguous","passed":sequence_ok},
    {"gate":"G12","requirement":"Response→Session→Turn relationships valid","passed":relations_ok},
    {"gate":"G13","requirement":"No silent many-to-many join expansion","passed":no_join_expansion_ok},
    {"gate":"G14","requirement":"Raw→canonical provenance valid","passed":traceability_ok},
    {"gate":"G15","requirement":"Serialization round-trip valid","passed":serialization_ok},
    {"gate":"G16","requirement":"Parser independent of target/test aggregates","passed":inference_symmetry_ok},
    {"gate":"G17","requirement":"Canonical files published only after audit PASS","passed":publication_after_pass_ok},
]
final_gate_df = pd.DataFrame(final_gate_records)
final_gate_df["status"] = np.where(final_gate_df["passed"], "PASS", "FAIL")
HARD_GATES_READY = bool(final_gate_df["passed"].all())

# Section-level gates for the manifest.
section_gate_df = pd.DataFrame([
    {"section":"3.0","gate":"INTEGRITY_BOOTSTRAP_READY","passed":INTEGRITY_BOOTSTRAP_READY},
    {"section":"3.1","gate":"ENTITY_KEY_INTEGRITY_READY","passed":ENTITY_KEY_INTEGRITY_READY},
    {"section":"3.2","gate":"RELATIONAL_INTEGRITY_READY","passed":RELATIONAL_INTEGRITY_READY},
    {"section":"3.3","gate":"STRUCTURAL_INTEGRITY_READY","passed":STRUCTURAL_INTEGRITY_READY},
    {"section":"3.4","gate":"DUPLICATE_OBJECTIVE_INTEGRITY_READY","passed":DUPLICATE_OBJECTIVE_INTEGRITY_READY},
    {"section":"3.5","gate":"FOLD_LEAKAGE_INTEGRITY_READY","passed":FOLD_LEAKAGE_INTEGRITY_READY},
    {"section":"3.6","gate":"TRACEABILITY_SYMMETRY_READY","passed":TRACEABILITY_SYMMETRY_READY},
    {"section":"3.7","gate":"CANONICAL_BUILD_READY","passed":CANONICAL_BUILD_READY},
    {"section":"3.8","gate":"CANONICAL_PUBLICATION_READY","passed":CANONICAL_PUBLICATION_READY},
])

INTEGRITY_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_WORKBOOK_PATH = INTEGRITY_DIR / "phase1_data_foundation_audit.xlsx"
PHASE1_MANIFEST_PATH = INTEGRITY_DIR / "phase1_manifest.json"
PHASE1_GATE_PATH = INTEGRITY_DIR / "phase1_gate.json"

turn_count_final = (
    int(pq.read_metadata(TURNS_CANDIDATE_PATH).num_rows)
    if PYARROW_READY and TURNS_CANDIDATE_PATH is not None and Path(TURNS_CANDIDATE_PATH).exists()
    else 0
)

phase_summary = pd.DataFrame([
    {"metric":"Responses","value":len(responses_base)},
    {"metric":"Sessions","value":len(sessions_candidate)},
    {"metric":"Turns","value":turn_count_final},
    {"metric":"Objectives","value":len(objectives_canonical)},
    {"metric":"Labelled response coverage","value":"100%" if coverage_ok else "FAIL"},
    {"metric":"Session cross-fold overlap","value":0 if zero_cross_fold_ok else "FAIL"},
    {"metric":"Parser deterministic","value":parser_deterministic_ok},
    {"metric":"Hard gates passed","value":int(final_gate_df["passed"].sum())},
    {"metric":"Hard gates total","value":len(final_gate_df)},
    {"metric":"Canonical publication ready","value":CANONICAL_PUBLICATION_READY},
    {"metric":"Pre-report phase status","value":"PASS" if HARD_GATES_READY else "FAIL"},
])

# Write audit workbook atomically.
workbook_ready = False
workbook_error = ""
audit_tmp = AUDIT_WORKBOOK_PATH.with_name("." + AUDIT_WORKBOOK_PATH.name + ".tmp.xlsx")
try:
    if audit_tmp.exists(): audit_tmp.unlink()
    sheets = {
        "00_PHASE_SUMMARY": phase_summary,
        "01_INPUT_LINEAGE": bootstrap_audit,
        "02_ENTITY_KEYS": entity_key_audit,
        "03_RELATION_COVERAGE": pd.concat(
            [relation_audit, relation_join_log] if len(relation_join_log) else [relation_audit],
            ignore_index=True, sort=False
        ),
        "04_STRUCTURE_QUALITY": pd.concat(
            [structure_audit, ordering_summary, text_quality_summary],
            ignore_index=True, sort=False
        ),
        "05_DUPLICATES_OBJECTIVES": pd.concat(
            [duplicate_objective_audit, duplicate_objective_summary],
            ignore_index=True, sort=False
        ),
        "06_FOLD_LEAKAGE": pd.concat(
            [fold_audit, fold_distribution], ignore_index=True, sort=False
        ),
        "07_TRACEABILITY_SYMMETRY": pd.concat(
            [traceability_audit, traceability_detail], ignore_index=True, sort=False
        ),
        "08_CANONICAL_ARTIFACTS": pd.concat(
            [canonical_build_audit, publication_audit, canonical_artifacts],
            ignore_index=True, sort=False
        ),
        "09_FINAL_GATE": final_gate_df,
    }
    with pd.ExcelWriter(audit_tmp, engine="openpyxl") as writer:
        for sheet, df in sheets.items():
            excel_safe_df(df).to_excel(writer, sheet_name=sheet[:31], index=False)
            ws = writer.book[sheet[:31]]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for col_cells in ws.columns:
                width = min(60, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col_cells) + 2))
                ws.column_dimensions[col_cells[0].column_letter].width = width
    os.replace(audit_tmp, AUDIT_WORKBOOK_PATH)
    workbook_ready = AUDIT_WORKBOOK_PATH.exists() and AUDIT_WORKBOOK_PATH.stat().st_size > 0
except Exception as e:
    workbook_error = f"{type(e).__name__}: {e}"
    try:
        audit_tmp.unlink(missing_ok=True)
    except Exception:
        pass

issues_df = pd.DataFrame(GLOBAL_ISSUES)
blocking_issue_count = 0 if issues_df.empty else int(issues_df["severity"].eq("BLOCKER").sum())
warning_count = 0 if issues_df.empty else int(issues_df["severity"].eq("WARNING").sum())
info_count = 0 if issues_df.empty else int(issues_df["severity"].eq("INFO").sum())

# Build manifest after workbook exists so its fingerprint is included.
manifest_payload = {
    "identity": {
        "project": "Trace the Ace",
        "phase": "01_data_foundation",
        "notebook": "03_data_integrity.ipynb",
        "notebook_version": NOTEBOOK_VERSION,
        "run_id": INTEGRITY_RUN_ID,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "status_before_report_files": "PASS" if HARD_GATES_READY else "FAIL",
    },
    "source_binding": {
        "data_contract_sha256": sha256_file(DATA_CONTRACT_PATH) if DATA_CONTRACT_PATH and DATA_CONTRACT_PATH.exists() else None,
        "inventory_recorded_data_contract_sha256": get_nested(inventory_manifest, ["artifacts","data_contract_sha256"]),
        "data_contract_exact_inventory_hash_match": bool(CONTRACT_HASH_EXACT),
        "data_contract_semantic_lineage_reconciled": bool(CONTRACT_LINEAGE_RECONCILED),
        "data_contract_lineage_detail": CONTRACT_LINEAGE_DETAIL,
        "inventory_manifest_sha256": sha256_file(INVENTORY_MANIFEST_PATH) if INVENTORY_MANIFEST_PATH and INVENTORY_MANIFEST_PATH.exists() else None,
        "parser_manifest_sha256": sha256_file(PARSER_MANIFEST_PATH) if PARSER_MANIFEST_PATH and PARSER_MANIFEST_PATH.exists() else None,
        "frozen_fold_manifest_sha256": sha256_file(FROZEN_FOLD_MANIFEST_PATH) if FROZEN_FOLD_MANIFEST_PATH and FROZEN_FOLD_MANIFEST_PATH.exists() else None,
        "responses_base_sha256": sha256_file(RESPONSES_BASE_PATH) if RESPONSES_BASE_PATH and RESPONSES_BASE_PATH.exists() else None,
        "turns_candidate_sha256": sha256_file(TURNS_CANDIDATE_PATH) if TURNS_CANDIDATE_PATH and TURNS_CANDIDATE_PATH.exists() else None,
        "sessions_candidate_sha256": sha256_file(SESSIONS_CANDIDATE_PATH) if SESSIONS_CANDIDATE_PATH and SESSIONS_CANDIDATE_PATH.exists() else None,
        "parser_payload_sha256": parser_manifest.get("manifest_payload_sha256"),
    },
    "population": {
        "responses": len(responses_base),
        "sessions": len(sessions_candidate),
        "turns": turn_count_final,
        "objectives": len(objectives_canonical),
    },
    "canonical_artifacts": canonical_artifacts.to_dict("records") if len(canonical_artifacts) else [],
    "audit_workbook": {
        "relative_path": AUDIT_WORKBOOK_PATH.relative_to(FOUNDATION_ROOT).as_posix() if workbook_ready else None,
        "sha256": sha256_file(AUDIT_WORKBOOK_PATH) if workbook_ready else None,
    },
    "section_gates": section_gate_df.to_dict("records"),
    "hard_gates": final_gate_df.to_dict("records"),
    "issues": GLOBAL_ISSUES,
    "issue_summary": {
        "blockers": blocking_issue_count,
        "warnings": warning_count,
        "info": info_count,
    },
    "candidate_to_canonical": {
        "turns_evidence_rewritten": False,
        "sessions_added_target_free_duplicate_groups": True,
        "responses_added_objective_uid": True,
        "objectives_include_target_statistics": False,
    },
}

manifest_ready, manifest_error = atomic_json_write(PHASE1_MANIFEST_PATH, manifest_payload)
manifest_hash = sha256_file(PHASE1_MANIFEST_PATH) if manifest_ready else None

PRE_GATE_FILE_READY = bool(HARD_GATES_READY and workbook_ready and manifest_ready)
gate_payload = {
    "project": "Trace the Ace",
    "phase": "01_data_foundation",
    "run_id": INTEGRITY_RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "PHASE_1_FOUNDATION_READY": PRE_GATE_FILE_READY,
    "hard_gates": final_gate_df.to_dict("records"),
    "hard_gates_passed": int(final_gate_df["passed"].sum()),
    "hard_gates_total": int(len(final_gate_df)),
    "blocking_failures": int((~final_gate_df["passed"]).sum()) + blocking_issue_count,
    "warnings": warning_count,
    "canonical_publication_ready": CANONICAL_PUBLICATION_READY,
    "audit_workbook_ready": workbook_ready,
    "phase1_manifest_ready": manifest_ready,
    "phase1_manifest_sha256": manifest_hash,
}
gate_ready, gate_error = atomic_json_write(PHASE1_GATE_PATH, gate_payload)

PHASE_1_FOUNDATION_READY = bool(PRE_GATE_FILE_READY and gate_ready)

report_status = pd.DataFrame([
    {"artifact":"phase1_data_foundation_audit.xlsx","ready":workbook_ready,
     "cause":workbook_error,"sha256":sha256_file(AUDIT_WORKBOOK_PATH) if workbook_ready else None},
    {"artifact":"phase1_manifest.json","ready":manifest_ready,
     "cause":manifest_error,"sha256":manifest_hash},
    {"artifact":"phase1_gate.json","ready":gate_ready,
     "cause":gate_error,"sha256":sha256_file(PHASE1_GATE_PATH) if gate_ready else None},
])

display(final_gate_df)
display(report_status)
if len(issues_df):
    print("\nWarnings / information retained for downstream diagnostics:")
    display(issues_df)

print("\n" + "=" * 82)
print("TRACE THE ACE — PHASE 1 FINAL HARD GATE")
print("=" * 82)
print(f"Responses                       : {len(responses_base):,}")
print(f"Sessions                        : {len(sessions_candidate):,}")
print(f"Turns                           : {turn_count_final:,}")
print(f"Objectives                      : {len(objectives_canonical):,}")
print(f"Hard gates passed               : {int(final_gate_df['passed'].sum())}/{len(final_gate_df)}")
print(f"Canonical publication ready     : {CANONICAL_PUBLICATION_READY}")
print(f"Audit workbook ready            : {workbook_ready}")
print(f"Phase manifest ready            : {manifest_ready}")
print(f"Phase gate file ready           : {gate_ready}")
print(f"PHASE 1 FOUNDATION READY        : {PHASE_1_FOUNDATION_READY}")
print("=" * 82)

if not PHASE_1_FOUNDATION_READY:
    print("\nPhase 1 is NOT frozen. Failed hard gates / report causes:")
    failed = final_gate_df.loc[~final_gate_df["passed"]]
    if len(failed): display(failed)
    failed_reports = report_status.loc[~report_status["ready"]]
    if len(failed_reports): display(failed_reports)
else:
    print("\nPhase 1 is frozen. Canonical data foundation is ready for architecture-driving diagnostics and retrieval.")

,gate,requirement,passed,status
0,G01,Current source/schema identity valid,True,PASS
1,G02,Feature-label response population aligned,True,PASS
2,G03,response_id unique,True,PASS
3,G04,Target valid,True,PASS
4,G05,Current authoritative objective population det...,True,PASS
5,G06,Labelled-session transcript coverage valid,True,PASS
6,G07,Every response assigned to exactly one frozen ...,True,PASS
7,G08,Zero session cross-fold overlap,True,PASS
8,G09,Parser deterministic,True,PASS
9,G10,Turn identity unique,True,PASS


,artifact,ready,cause,sha256
0,phase1_data_foundation_audit.xlsx,True,,5be7d5e30a86bad2968c87e343801aa02d5384343f3240...
1,phase1_manifest.json,True,,5317a9f1bf6b4dd98ec4d3a0e005684126e6a2ed747758...
2,phase1_gate.json,True,,449ea097173f4df5a115c72cd710c9f64b291a96c97afc...



Warnings / information retained for downstream diagnostics:


,section,code,severity,count,detail
0,3.0,STALE_INVENTORY_DATA_CONTRACT_HASH,WARNING,1,inventory=a1862c9a5ad31287e3e3c7fff550c2e5cf30...
1,3.0,PARSER_CONTRACT_VERSION_NOT_RECORDED,INFO,1,Compatibility is established from certified pa...
2,3.3,UNCLEAR_MARKER_PRESENT,INFO,180538,Original evidence retained; no deletion.
3,3.3,VERY_LONG_UTTERANCE,INFO,474,Useful Phase-2 context-length diagnostic; not ...
4,3.3,CONSECUTIVE_EXACT_CONTENT,WARNING,58061,Preserved as evidence; inspect during advanced...



TRACE THE ACE — PHASE 1 FINAL HARD GATE
Responses                       : 35,072
Sessions                        : 22,821
Turns                           : 6,139,854
Objectives                      : 398
Hard gates passed               : 17/17
Canonical publication ready     : True
Audit workbook ready            : True
Phase manifest ready            : True
Phase gate file ready           : True
PHASE 1 FOUNDATION READY        : True

Phase 1 is frozen. Canonical data foundation is ready for architecture-driving diagnostics and retrieval.
